# Library and settings

In [1]:
import os
os.chdir("..")  # Let the wd be outside

In [2]:
from src.utility import *

import datasets
import duckdb
import gdown
from huggingface_hub import notebook_login, snapshot_download
import json
from kagglehub import dataset_download
import numpy as np
import pandas as pd
import pooch
import tldextract
from typing import Union, List
import zipfile

In [3]:
RUN_ALL = True

# Data processing

## Datasets hosted on GitHub

In [4]:
# We will use Pooch. See https://www.fatiando.org/pooch/dev/index.html

# Dict of filename-hash value pairs for datasets.
github_urls = {
    "news-media-reliability_acl2020_corpus.tsv": "ramybaly/News-Media-Reliability/raw/master/data/acl2020/corpus.tsv",
    "news-media-reliability_emnlp18_corpus.tsv": "ramybaly/News-Media-Reliability/raw/master/data/emnlp18/corpus.tsv",
    "domain-quality-ratings_domain_ratings.csv": "hauselin/domain-quality-ratings/raw/main/data/domain_ratings.csv",
    "fakenewscorpus.zip": "architapathak/FakeNewsCorpus/raw/master/ACL2019_Data.zip",
    "fakehealth_release.json": "EnyanDai/FakeHealth/raw/master/dataset/reviews/HealthRelease.json",
    "fakehealth_story.json": "EnyanDai/FakeHealth/raw/master/dataset/reviews/HealthStory.json",
    "recovery.csv": "apurvamulay/ReCOVery/raw/master/dataset/recovery-news-data.csv",
    "politifact_fake.csv": "KaiDMML/FakeNewsNet/raw/master/dataset/politifact_fake.csv",
    "politifact_real.csv": "KaiDMML/FakeNewsNet/raw/master/dataset/politifact_real.csv",
    "gossipcop_fake.csv": "KaiDMML/FakeNewsNet/raw/master/dataset/gossipcop_fake.csv",
    "gossipcop_real.csv": "KaiDMML/FakeNewsNet/raw/master/dataset/gossipcop_real.csv",
    "test2.tsv": "Tariq60/LIAR-PLUS/raw/master/dataset/tsv/test2.tsv",
    "train2.tsv": "Tariq60/LIAR-PLUS/raw/master/dataset/tsv/train2.tsv",
    "val2.tsv": "Tariq60/LIAR-PLUS/raw/master/dataset/tsv/val2.tsv",
}

github_registry = {
    "ramybaly/News-Media-Reliability/raw/master/data/acl2020/corpus.tsv": "936cc629df930c7caae8b8506fe3843a6b0576c4f4afcd4b371b0156e8a65515",
    "ramybaly/News-Media-Reliability/raw/master/data/emnlp18/corpus.tsv": "7f96b15459a23bc5267ca5dc0df66e71d0dc38a28b797b2ec3d228b818427cbf",
    "hauselin/domain-quality-ratings/raw/main/data/domain_ratings.csv": "1cf331996d880ae0912c64950a5d9aa8101350a380d4f1c8a81bfab663adc8fd",
    "architapathak/FakeNewsCorpus/raw/master/ACL2019_Data.zip": None,  # Skip this hash-check, since it requires unzipping logic
    "EnyanDai/FakeHealth/raw/master/dataset/reviews/HealthRelease.json": "f0c800154e29f1df8599b91c337ae362641083aa44e3c93f3a75186b861dcd1f",
    "EnyanDai/FakeHealth/raw/master/dataset/reviews/HealthStory.json": "4d9db2ec053888234313e8ef9a8c999fe9b57cbc3dea8c0181c217827256fd7c",
    "apurvamulay/ReCOVery/raw/master/dataset/recovery-news-data.csv": "9b63044e77d4a8d9b8e1d283daf63a338ef3dd4c28bd2ac10121759aabc74982",
    "KaiDMML/FakeNewsNet/raw/master/dataset/politifact_fake.csv": "abe7fe7aad801b1e2ab5fc963cbd0881d9670190262ae3df85e91c3380bd004c",
    "KaiDMML/FakeNewsNet/raw/master/dataset/politifact_real.csv": "2500f86a7addca0f59fe8cf089c0f802d3eca6980f53538992b3b0e9687dbae0",
    "KaiDMML/FakeNewsNet/raw/master/dataset/gossipcop_fake.csv": "c6932bffadb1230bc61c66bb2ca298f4ef28ce849d268313ad2eed41d18d4c99",
    "KaiDMML/FakeNewsNet/raw/master/dataset/gossipcop_real.csv": "d721e9a8b7e660da05c089691743c33fecb25367074e936b5fab3d85cdcbe79f",
    "Tariq60/LIAR-PLUS/raw/master/dataset/tsv/val2.tsv": "45fac5748e2da08898a4fda0cb02e4c107990a0111b4a6c3aed1956b38a309a2",
    "Tariq60/LIAR-PLUS/raw/master/dataset/tsv/train2.tsv": "63adafa9360f43788fb16499871ff7665143c554fc8a3cce1af846233ce53f07",
    "Tariq60/LIAR-PLUS/raw/master/dataset/tsv/test2.tsv": "a374044e6f73700f749a1bdaf71174356e437c7e6f7ecd4feb52d2fa82820442",
}

github_dog = pooch.create(
    pooch.os_cache("thesis"),
    "https://github.com/",
    registry=github_registry,
    urls=github_urls,
    allow_updates=False,
)

for auto_fname, url in github_dog.urls.items():
    if url.endswith(".zip"):
        github_dog.fetch(url, processor=pooch.Unzip())
    else:
        fname = github_dog.fetch(url)
        print(f"...{fname[-30:]} has hash {pooch.file_hash(fname)}")

...master\data\acl2020\corpus.tsv has hash 936cc629df930c7caae8b8506fe3843a6b0576c4f4afcd4b371b0156e8a65515


...master\data\emnlp18\corpus.tsv has hash 7f96b15459a23bc5267ca5dc0df66e71d0dc38a28b797b2ec3d228b818427cbf


...w\main\data\domain_ratings.csv has hash 1cf331996d880ae0912c64950a5d9aa8101350a380d4f1c8a81bfab663adc8fd


Unzipping contents of 'C:\Users\tharg\AppData\Local\thesis\thesis\Cache\architapathak\FakeNewsCorpus\raw\master\ACL2019_Data.zip' to 'C:\Users\tharg\AppData\Local\thesis\thesis\Cache\architapathak\FakeNewsCorpus\raw\master\ACL2019_Data.zip.unzip'


...set\reviews\HealthRelease.json has hash f0c800154e29f1df8599b91c337ae362641083aa44e3c93f3a75186b861dcd1f


...

...er\dataset\politifact_real.csv has hash 2500f86a7addca0f59fe8cf089c0f802d3eca6980f53538992b3b0e9687dbae0


...ter\dataset\gossipcop_fake.csv has hash c6932bffadb1230bc61c66bb2ca298f4ef28ce849d268313ad2eed41d18d4c99


...ter\dataset\gossipcop_real.csv has hash d721e9a8b7e660da05c089691743c33fecb25367074e936b5fab3d85cdcbe79f


...w\master\dataset\tsv\test2.tsv has hash a374044e6f73700f749a1bdaf71174356e437c7e6f7ecd4feb52d2fa82820442


...\master\dataset\tsv\train2.tsv has hash 63adafa9360f43788fb16499871ff7665143c554fc8a3cce1af846233ce53f07
...aw\master\dataset\tsv\val2.tsv has hash 45fac5748e2da08898a4fda0cb02e4c107990a0111b4a6c3aed1956b38a309a2


## Datasets hosted by HuggingFace, but not as HuggingFace-Datasets

In [5]:
# Dict of filename-hash value pairs for datasets.
huggingface_urls = {
    "averitec_train.json": "chenxwh/AVeriTeC/resolve/main/data/train.json",
    "averitec_test.json": "chenxwh/AVeriTeC/resolve/main/data/test.json",
    "averitec_dev.json": "chenxwh/AVeriTeC/resolve/main/data/dev.json",
}

huggingface_registry = {
    "chenxwh/AVeriTeC/resolve/main/data/train.json": "ae5eda7c42ddf1695ef185a7ba1bc716928f5adf57103e4f78aae5f9afe00f9c",
    "chenxwh/AVeriTeC/resolve/main/data/test.json": "c62af19526ebc00fdca21ea857282f061326a247ac265c93d55ab6765defbe49",
    "chenxwh/AVeriTeC/resolve/main/data/dev.json": "499793726b4a5406780928a3d9dedc48d6dd53de778f22437d129cacdb08e300",
}

huggingface_dog = pooch.create(
    pooch.os_cache("thesis"),
    "https://huggingface.co/",
    registry=huggingface_registry,
    urls=huggingface_urls,
    allow_updates=False,
)

for auto_fname, url in huggingface_dog.urls.items():
    fname = huggingface_dog.fetch(url)
    print(f"...{fname[-30:]} has hash {pooch.file_hash(fname)}")

...C\resolve\main\data\train.json has hash ae5eda7c42ddf1695ef185a7ba1bc716928f5adf57103e4f78aae5f9afe00f9c


...eC\resolve\main\data\test.json has hash c62af19526ebc00fdca21ea857282f061326a247ac265c93d55ab6765defbe49
...TeC\resolve\main\data\dev.json has hash 499793726b4a5406780928a3d9dedc48d6dd53de778f22437d129cacdb08e300


## Datasets hosted on HuggingFace Datasets

In [6]:
druid_plus = datasets.load_dataset(
    "copenlu/druid", "DRUID+", token=HF_HUB_TOKEN, split="train"
)

README.md: 0.00B [00:00, ?B/s]

c:\Users\tharg\anaconda3\envs\gnn_cf\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tharg\.cache\huggingface\hub\datasets--copenlu--druid. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


druid_plus.tsv:   0%|          | 0.00/55.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/48014 [00:00<?, ? examples/s]

## Datasets hosted on Kaggle

In [7]:
covid19_fake_news_path = dataset_download("arashnic/covid19-fake-news")
politifact_fact_check_path = dataset_download(
    "rmisra/politifact-fact-check-dataset"
)
# The saved Kaggle cache path contains a single file.
politifact_fact_check_path = os.path.join(
    politifact_fact_check_path, os.listdir(politifact_fact_check_path)[0]
)

# Remove some unnecessary datafiles
for path in os.listdir(covid19_fake_news_path):
    if "tweets" in path:
        os.remove(os.path.join(covid19_fake_news_path, path))

## Other datasets

In [8]:
multifc = pooch.retrieve(
    "https://www.copenlu.com/publication/2019_emnlp_augenstein/MultiFC_Codalab.zip",
    known_hash="92aa1563ae21c14b9bb377b468976f774ae3287c94321f752fd785b28134cb4f",
    path=pooch.os_cache("thesis"),
    processor=pooch.Unzip(
        members=[
            "public_data.zip",
            "reference_multi_fc.zip",
            "scoring_multifc.zip",
        ]
    ),
)

mm_covid19_english_news = gdown.download(
    "https://drive.google.com/uc?export=download&id=13dSxwMPrN53YHwvq_D_UPkj6AymMclQW",
    str(
        pooch.os_cache("thesis").joinpath("mm_covid19_english_news.csv")
    ),  # Needs to be string for some reason
    quiet=False,
    resume=True,
)


# This is a nested zipfile, so unzip the rest here
for path in multifc:
    if path.endswith(".zip"):  # This check is only necessarry on repeat runs
        with zipfile.ZipFile(path) as z:
            z.extractall(path + ".unzip")

Extracting 'public_data.zip' from 'C:\Users\tharg\AppData\Local\thesis\thesis\Cache\ec43320311f9d0e79384f2a034e1291f-MultiFC_Codalab.zip' to 'C:\Users\tharg\AppData\Local\thesis\thesis\Cache\ec43320311f9d0e79384f2a034e1291f-MultiFC_Codalab.zip.unzip'
Extracting 'reference_multi_fc.zip' from 'C:\Users\tharg\AppData\Local\thesis\thesis\Cache\ec43320311f9d0e79384f2a034e1291f-MultiFC_Codalab.zip' to 'C:\Users\tharg\AppData\Local\thesis\thesis\Cache\ec43320311f9d0e79384f2a034e1291f-MultiFC_Codalab.zip.unzip'
Extracting 'scoring_multifc.zip' from 'C:\Users\tharg\AppData\Local\thesis\thesis\Cache\ec43320311f9d0e79384f2a034e1291f-MultiFC_Codalab.zip' to 'C:\Users\tharg\AppData\Local\thesis\thesis\Cache\ec43320311f9d0e79384f2a034e1291f-MultiFC_Codalab.zip.unzip'
Downloading...
From (original): https://drive.google.com/uc?export=download&id=13dSxwMPrN53YHwvq_D_UPkj6AymMclQW
From (redirected): https://drive.google.com/uc?export=download&id=13dSxwMPrN53YHwvq_D_UPkj6AymMclQW&confirm=t&uuid=0251f81c

Be aware that Snopes, PolitiFact and Emergent.Info data must be manually scraped and moved using http://fakenews.research.sfu.ca/#parseWebs !

## Create single Pooch instance for watching all files not managed by KaggleHub or HuggingFaceHub

In [9]:
registry_path = pooch.os_cache("thesis").joinpath("registry.txt")
pooch.make_registry(pooch.os_cache("thesis"), registry_path, recursive=True)
# Currently stores the registry within the Cache, hence it can become registered within itself. Change this if issues arise.

dog = pooch.create(pooch.os_cache("thesis"), base_url="")
dog.load_registry(registry_path)
# Its registry is more bloated, because it is recursively fetched

# Create a lookup table with more readable filenames
file_aliases = {
    "averitec_dev.json": "chenxwh/AVeriTeC/resolve/main/data/dev.json",
    "averitec_test.json": "chenxwh/AVeriTeC/resolve/main/data/test.json",
    "averitec_train.json": "chenxwh/AVeriTeC/resolve/main/data/train.json",
    "emergent.csv": "emergent_phase1_clean_2018_7_2/emergent_phase1_clean_2018_7_2.csv",
    "domain_ratings.csv": "hauselin/domain-quality-ratings/raw/main/data/domain_ratings.csv",
    "fake_news_corpus.json": "architapathak/FakeNewsCorpus/raw/master/ACL2019_Data.zip.unzip/ACL2019_Data/Fakes700.json",
    "health_release.json": "EnyanDai/FakeHealth/raw/master/dataset/reviews/HealthRelease.json",
    "health_story.json": "EnyanDai/FakeHealth/raw/master/dataset/reviews/HealthStory.json",
    "multifc_public_data_dev.tsv": "ec43320311f9d0e79384f2a034e1291f-MultiFC_Codalab.zip.unzip/public_data.zip.unzip/dev.tsv",
    "multifc_public_data_test.tsv": "ec43320311f9d0e79384f2a034e1291f-MultiFC_Codalab.zip.unzip/public_data.zip.unzip/test.tsv",
    "multifc_public_data_train.tsv": "ec43320311f9d0e79384f2a034e1291f-MultiFC_Codalab.zip.unzip/public_data.zip.unzip/train.tsv",
    "multifc_reference_labels.txt": "ec43320311f9d0e79384f2a034e1291f-MultiFC_Codalab.zip.unzip/reference_multi_fc.zip.unzip/labels.txt",
    "news_media_reliability_acl": "ramybaly/News-Media-Reliability/raw/master/data/acl2020/corpus.tsv",
    "news_media_reliability_emnlp": "ramybaly/News-Media-Reliability/raw/master/data/emnlp18/corpus.tsv",
    "politifact.csv": "politifact_phase1_clean_2018_7_3/politifact_phase1_clean_2018_7_3.csv",
    "recovery.csv": "apurvamulay/ReCOVery/raw/master/dataset/recovery-news-data.csv",
    "snopes.csv": "snopes_phase1_clean_2018_7_3/snopes_phase1_clean_2018_7_3.csv",
    "politifact_fact_check_fake.csv": "KaiDMML/FakeNewsNet/raw/master/dataset/politifact_fake.csv",
    "politifact_fact_check_real.csv": "KaiDMML/FakeNewsNet/raw/master/dataset/politifact_real.csv",
    "gossipcop_fact_check_fake.csv": "KaiDMML/FakeNewsNet/raw/master/dataset/gossipcop_fake.csv",
    "gossipcop_fact_check_real.csv": "KaiDMML/FakeNewsNet/raw/master/dataset/gossipcop_real.csv",
    "liar_plus_test.tsv": "Tariq60/LIAR-PLUS/raw/master/dataset/tsv/test2.tsv",
    "liar_plus_train.tsv": "Tariq60/LIAR-PLUS/raw/master/dataset/tsv/train2.tsv",
    "liar_plus_val.tsv": "Tariq60/LIAR-PLUS/raw/master/dataset/tsv/val2.tsv",
    "mm_covid19.csv": "mm_covid19_english_news.csv",
}


def dogpath(alias):
    return dog.fetch(file_aliases[alias])

In [10]:
# Clean up the registry. Ad hoc.
def discardable(path):
    return any(
        [
            "Fakes700/article" in path,
            "reference_multi_fc" in path,
            any(
                [
                    path.endswith(suffix)
                    for suffix in [
                        ".zip",
                        "test.solution",
                        "score.py",
                        "registry.txt",
                    ]
                ]
            ),
        ]
    )


topop = [path for path in dog.registry.keys() if discardable(path)]
for poppable in topop:
    dog.registry.pop(poppable)

# Inspect datasets

## AVeriTeC

In [11]:
averitec = datasets.load_dataset(
    "json",
    data_files=[dogpath("averitec_dev.json"), dogpath("averitec_train.json")],
    split="train",
)
averitec

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['claim', 'required_reannotation', 'label', 'justification', 'claim_date', 'speaker', 'original_claim_url', 'fact_checking_article', 'reporting_source', 'location_ISO_code', 'claim_types', 'fact_checking_strategies', 'questions', 'cached_original_claim_url'],
    num_rows: 3568
})

In [12]:
# Add a column with answer types. dataset.flatten() does not work here.
def extract_answer_type_column(row):
    row["answer_types"] = [
        answer["answer_type"]
        for question in row["questions"]
        for answer in question["answers"]
    ]
    return row


averitec = averitec.map(lambda row: extract_answer_type_column(row))
averitec

Map:   0%|          | 0/3568 [00:00<?, ? examples/s]

Dataset({
    features: ['claim', 'required_reannotation', 'label', 'justification', 'claim_date', 'speaker', 'original_claim_url', 'fact_checking_article', 'reporting_source', 'location_ISO_code', 'claim_types', 'fact_checking_strategies', 'questions', 'cached_original_claim_url', 'answer_types'],
    num_rows: 3568
})

In [13]:
averitec_df = averitec.to_pandas()
duplicate_claims_df = averitec_df[averitec_df["claim"].duplicated()]
duplicate_claims_df

,claim,required_reannotation,label,justification,claim_date,speaker,original_claim_url,fact_checking_article,reporting_source,location_ISO_code,claim_types,fact_checking_strategies,questions,cached_original_claim_url,answer_types
205,Nigeria’s urban population at independence was...,False,Supported,The second answer and source show that the pop...,1-10-2020,Muhammadu Buhari,https://web.archive.org/web/20210118154309/htt...,https://web.archive.org/web/20210526202243/htt...,Channels Television,NG,[Numerical Claim],[Written Evidence],"[{'answers': [{'answer': 'October 1st 1960', '...",https://web.archive.org/web/20210118154309/htt...,"[Abstractive, Abstractive]"
207,"At independence, Nigeria had a population of 4...",False,Supported,From the 2nd QA it seems thatthe number is acc...,1-10-2020,Muhammadu Buhari,https://web.archive.org/web/20210118154309/htt...,https://web.archive.org/web/20210526202243/htt...,None,NG,[Numerical Claim],[Written Evidence],[{'answers': [{'answer': 'Nigeria achieved ind...,https://web.archive.org/web/20210118154309/htt...,"[Extractive, Extractive]"
208,It makes no sense for oil to be cheaper in Nig...,False,Not Enough Evidence,All of the evidence presented relates to Niger...,1-10-2020,Muhammadu Buhari,https://web.archive.org/web/20210118154309/htt...,https://web.archive.org/web/20210526202243/htt...,None,NG,[Numerical Claim],[Written Evidence],[{'answers': [{'answer': 'Fuel prices can be b...,https://web.archive.org/web/20210118154309/htt...,"[Extractive, Extractive, Extractive, Extractiv..."
211,"At independence, Nigeria had a population of 4...",False,Supported,This claim is supported. According to the QA p...,1-10-2020,Muhammadu Buhari,https://web.archive.org/web/20210118154309/htt...,https://web.archive.org/web/20210526202243/htt...,None,NG,[Numerical Claim],[Written Evidence],"[{'answers': [{'answer': '1960', 'answer_type'...",https://web.archive.org/web/20210118154309/htt...,"[Abstractive, Abstractive]"
212,For a cumulative 29 of our 60 years of existen...,False,Supported,Up to the claim date Nigeria had been independ...,1-10-2020,Muhammadu Buhari,https://web.archive.org/web/20210118154309/htt...,https://web.archive.org/web/20210526202243/htt...,None,NG,[Numerical Claim],[Written Evidence],[{'answers': [{'answer': 'Nigeria returned to ...,https://web.archive.org/web/20210118154309/htt...,"[Extractive, Extractive]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3187,"In 2017, South Africa recorded an inflow of fo...",True,Supported,The QA pair shows that the amount although in ...,7-2-2019,Cyril Ramaphosa,None,https://web.archive.org/web/20210226001653/htt...,None,ZA,[Numerical Claim],[Numerical Comparison],[{'answers': [{'answer': 'Foreign direct inves...,None,"[Extractive, Extractive, Extractive, Extractiv..."
3203,"Over 700,000 children [accessed] early childho...",True,Conflicting Evidence/Cherrypicking,"while the claim is technically correct, as the...",7-2-2019,Cyril Ramaphosa,None,https://web.archive.org/web/20210226001653/htt...,None,ZA,[Numerical Claim],[Numerical Comparison],[{'answers': [{'answer': 'yes. Ramaphosa put t...,None,"[Abstractive, Boolean, Abstractive, Extractive..."
3209,"Over the past five years, we’ve made significa...",True,Not Enough Evidence,Claim says over past 5 years but QA pairs stat...,7-2-2019,Cyril Ramaphosa,None,https://web.archive.org/web/20210226001653/htt...,None,ZA,[Numerical Claim],[Numerical Comparison],"[{'answers': [{'answer': 'the year 2014/2015',...",None,"[Abstractive, Extractive, Extractive]"
3213,Every month 17.5 million social grants are pro...,True,Supported,The sources proved that each month around 17.5...,7-2-2019,Cyril Ramaphosa,None,https://web.archive.org/web/20210226001653/htt...,State of the Nation Address,ZA,[Numerical Claim],"[Numerical Comparison, Consultation]",[{'answers': [{'answer': 'South African Social...,None,"[Abstractive, Extractive, Boolean]"


In [14]:
set(averitec_df.claim_date.str[-4:])

{'1919',
 '1920',
 '1921',
 '1922',
 '2008',
 '2009',
 '2011',
 '2012',
 '2013',
 '2014',
 '2015',
 '2016',
 '2017',
 '2018',
 '2019',
 '2020',
 None}

- 3,568 rows
- 3.486 unique claims. Multiple reasons for duplicates. Some attribute the same statement to multiple speakers, sometimes hierarchically, e.g. to a person and to their organization. Sometimes, multiple answers i.e. fact-checks exist for the same claim.
- "Speaker" column with names, 44% of which are unique and 22% of which are missing
- "fact_checking_article" column with URLs via web.archive from many different sources. 2368 values.
- "fact_checking_strategies" string-list column
- Labels in {Not Enough Evidence, Supported, Refuted, Conflicting Evidence/Cherrypicking}
- "answer_type" attributes in a nested JSON in the "questions" column.

In [15]:
averitec_test = datasets.load_dataset(
    "json", data_files=dogpath("averitec_test.json"), split="train"
)
averitec_test

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['claim', 'claim_id', 'claim_date', 'speaker', 'original_claim_url', 'reporting_source', 'location_ISO_code'],
    num_rows: 2215
})

The test set does not contain labels.

In [16]:
# Let's drop the "Not Enough Evidence"-labeled rows, they don't fit neatly
# downstream.
averitec = averitec.filter(lambda row: row["label"] != "Not Enough Evidence")
averitec

Filter:   0%|          | 0/3568 [00:00<?, ? examples/s]

Dataset({
    features: ['claim', 'required_reannotation', 'label', 'justification', 'claim_date', 'speaker', 'original_claim_url', 'fact_checking_article', 'reporting_source', 'location_ISO_code', 'claim_types', 'fact_checking_strategies', 'questions', 'cached_original_claim_url', 'answer_types'],
    num_rows: 3251
})

In [17]:
dir(averitec)

['_TF_DATASET_REFS',
 '__class__',
 '__del__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__enter__',
 '__eq__',
 '__exit__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getitems__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__slotnames__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_build_local_temp_path',
 '_check_index_is_initialized',
 '_data',
 '_estimate_nbytes',
 '_fingerprint',
 '_format_columns',
 '_format_kwargs',
 '_format_type',
 '_generate_tables_from_cache_file',
 '_generate_tables_from_shards',
 '_get_cache_file_path',
 '_get_output_signature',
 '_getitem',
 '_indexes',
 '_indices',
 '_info',
 '_map_single',
 '_new_dataset_with_indices',
 '_output_all_columns',
 '_push_par

## Breaking! Fake News Dataset

In [18]:
fake_news_corpus = datasets.load_dataset(
    "json", data_files=dogpath("fake_news_corpus.json"), split="train"
)
fake_news_corpus

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['id', 'authors', 'headline', 'type', 'urls'],
    num_rows: 705
})

- Headlines. "author" column contains both people and organisations. Contains repeat and missing authors. Also contains a "urls" column with the stripped URL!
- "type" column of only 0.97 questionable and 0.03 fake, i.e. <u>a negative dataset!</u>

## Covid Fake News

### Claims

In [19]:
covid19_fake_claims = datasets.load_dataset(
    "csv",
    data_files=[
        os.path.join(covid19_fake_news_path, path)
        for path in os.listdir(covid19_fake_news_path)
        if "ClaimFake" in path
    ],
    split="train",
)
covid19_fake_claims = add_binary_label_column(covid19_fake_claims, False)
covid19_real_claims = datasets.load_dataset(
    "csv",
    data_files=[
        os.path.join(covid19_fake_news_path, path)
        for path in os.listdir(covid19_fake_news_path)
        if "ClaimReal" in path
    ],
    split="train",
)
covid19_real_claims = add_binary_label_column(covid19_real_claims, True)
covid19_claims = datasets.concatenate_datasets(
    [covid19_fake_claims, covid19_real_claims]
)
covid19_claims

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['Unnamed: 0', 'fact_check_url', 'news_url', 'title', 'label'],
    num_rows: 482
})

- Only 482 rows
- Binary labels
- Source is "news_url" column. This is only a small set of fact-check URLs, hence do not indicate sources.
- <u>Do not use this dataset.</u>

### News

In [20]:
covid19_fake_news = datasets.load_dataset(
    "csv",
    data_files=[
        os.path.join(covid19_fake_news_path, path)
        for path in os.listdir(covid19_fake_news_path)
        if "NewsFake" in path
    ],
    split="train",
)
covid19_fake_news = add_binary_label_column(covid19_fake_news, False)
covid19_real_news = datasets.load_dataset(
    "csv",
    data_files=[
        os.path.join(covid19_fake_news_path, path)
        for path in os.listdir(covid19_fake_news_path)
        if "NewsReal" in path
    ],
    split="train",
)
covid19_real_news = add_binary_label_column(covid19_real_news, True)
covid19_news = datasets.concatenate_datasets(
    [covid19_fake_news, covid19_real_news]
)
covid19_news

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['Unnamed: 0', 'type', 'fact_check_url', 'archieve', 'news_url', 'news_url2', 'news_url3', 'news_url4', 'news_url5', 'title', 'newstitle', 'content', 'abstract', 'publish_date', 'meta_keywords', 'label'],
    num_rows: 4504
})

In [21]:
if RUN_ALL:
    retrieve_heterogeneously_formatted_years(covid19_news["publish_date"])

- 4504 rows.
- Binary labels.
- 1:5 ratio of false to true labels.
- "type" column for distinguishing social media "post"s from "article"s.
- Contains "fact_check_url" and "news_url({2-5})" for the different URLs cited in the fact-check, i.e. often instagram, reddit, facebook etc.
- Very messy dataset. Missing values, duplicates, unsuccessful scrapes (e.g. scraping login-portals rather than articles).
- Contains archival links for some fact checks.

## Domain Ratings

In [22]:
domain_ratings = datasets.load_dataset(
    "csv", data_files=dogpath("domain_ratings.csv"), split="train"
)
domain_ratings

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['domain', 'pc1', 'afm', 'afm_bias', 'afm_min', 'afm_rely', 'fc', 'mbfc', 'mbfc_bias', 'mbfc_fact', 'mbfc_min', 'lewandowsky_acc', 'lewandowsky_trans', 'lewandowsky_rely', 'lewandowsky_mean', 'lewandowsky_min', 'misinfome_bin'],
    num_rows: 11520
})

- 11,520 rows.
- Ratings from a series of outlets per domain, as well as a derived PC1 value and some statistical values. All ratings are normalized to [0, 1]. Ratings are from NewsGuard, {bias, factuality, final score and minimum} from MBFC, {bias, reliability and derived mean- and minimum} from Ad Fontes Media, >1K Mechanical Turk crowdsourced fact-checkers from the Pennycook paper as well as 60 by professionals jointly labeled as "fact-checkers", the dataset of Lasser et al. <u>which itself is from NewsGuard</u>, and finally, a score derived by assigning 0 to sources from the [Iffy index](https://iffy.news) and otherwise 1, hence potentially containing false positives.
- Not all statistical values apply to each row (presumably based on whether they are motivated given the set of ratings)

## DRUID+

In [23]:
druid_plus

Dataset({
    features: ['id', 'claim_id', 'claim_source', 'claim', 'claimant', 'claim_date', 'evidence_source', 'evidence', 'evidence_date', 'factcheck_verdict', 'is_gold', 'relevant', 'evidence_stance'],
    num_rows: 48014
})

In [24]:
druid_plus_df = druid_plus.to_pandas()
druid_plus_df["claimant"].unique()

array(['Abyei', 'Bassas da India', 'Aarsal', 'Deir El Aachayer',
       'Kfar Qouq', 'Mazraat Deir al-Ashayer', 'Qaa', 'Qasr',
       'Europa Island', 'Tuffah', 'Shatt al-Arab', 'Abu Musa',
       'Greater and Lesser Tunbs', 'Nagorno-Karabakh region',
       'Syunik Province', 'Azad Kashmir', 'Gilgit-Baltistan',
       'Jammu and Kashmir', 'Aksai Chin', 'Juan de Nova Island', 'Tawang',
       'Arunachal Pradesh', 'David Gareja monastery complex', 'Doi Lang',
       'Fasht Ad Dibal', "Qit'at Jaradah", 'Fergana Valley',
       'Isfara Valley', 'Ceuta', 'Ambalat', 'Kabaw', 'Golan Heights',
       'Green Line', 'West Bank', 'East Jerusalem', 'Kalapani',
       'Susta River', 'Antudanda', 'Nawalparasi', 'Melilla', 'Artsvashen',
       'Gegharkunik province', 'Karki', 'Nakhichevan Autonomous Republic',
       'Yukhari Askipara', 'Barkhudarli', 'Qazakh Rayon', 'Yaradullu',
       'Khuriya Muriya Islands', 'Korean Peninsula',
       'plazas de soberanía', 'Military Demarcation Line',
       'S

In [25]:
sorted(druid_plus_df["claim_date"].str[:7].unique())

['2013-03',
 '2014-02',
 '2015-01',
 '2015-02',
 '2015-07',
 '2015-11',
 '2015-12',
 '2016-01',
 '2016-03',
 '2016-04',
 '2016-05',
 '2016-06',
 '2016-07',
 '2016-08',
 '2016-11',
 '2016-12',
 '2017-01',
 '2017-02',
 '2017-03',
 '2017-04',
 '2017-05',
 '2017-06',
 '2017-07',
 '2017-08',
 '2017-09',
 '2017-10',
 '2017-11',
 '2018-01',
 '2018-02',
 '2018-03',
 '2018-04',
 '2018-05',
 '2018-06',
 '2018-07',
 '2018-08',
 '2018-09',
 '2018-10',
 '2018-11',
 '2018-12',
 '2019-01',
 '2019-02',
 '2019-03',
 '2019-04',
 '2019-05',
 '2019-06',
 '2019-07',
 '2019-08',
 '2019-09',
 '2019-10',
 '2019-11',
 '2019-12',
 '2020-01',
 '2020-02',
 '2020-03',
 '2020-04',
 '2020-05',
 '2020-06',
 '2020-07',
 '2020-08',
 '2020-09',
 '2020-10',
 '2020-11',
 '2020-12',
 '2021-01',
 '2021-02',
 '2021-03',
 '2021-04',
 '2021-05',
 '2021-06',
 '2021-07',
 '2021-08',
 '2021-09',
 '2021-10',
 '2021-11',
 '2021-12',
 '2022-01',
 '2022-02',
 '2022-03',
 '2022-04',
 '2022-05',
 '2022-06',
 '2022-07',
 '2022-08',
 '20

In [26]:
for prefix in (
    druid_plus_df["claim_id"].str.partition("-")[0].str.partition("_")[0].unique()
):
    subset = druid_plus_df[druid_plus_df["claim_id"].str.startswith(prefix)]
    print(prefix)
    print(subset.claimant.unique())
    print(subset.factcheck_verdict.unique(), end="\n\n")

borderlines
['Abyei' 'Bassas da India' 'Aarsal' 'Deir El Aachayer' 'Kfar Qouq'
 'Mazraat Deir al-Ashayer' 'Qaa' 'Qasr' 'Europa Island' 'Tuffah'
 'Shatt al-Arab' 'Abu Musa' 'Greater and Lesser Tunbs'
 'Nagorno-Karabakh region' 'Syunik Province' 'Azad Kashmir'
 'Gilgit-Baltistan' 'Jammu and Kashmir' 'Aksai Chin' 'Juan de Nova Island'
 'Tawang' 'Arunachal Pradesh' 'David Gareja monastery complex' 'Doi Lang'
 'Fasht Ad Dibal' "Qit'at Jaradah" 'Fergana Valley' 'Isfara Valley'
 'Ceuta' 'Ambalat' 'Kabaw' 'Golan Heights' 'Green Line' 'West Bank'
 'East Jerusalem' 'Kalapani' 'Susta River' 'Antudanda' 'Nawalparasi'
 'Melilla' 'Artsvashen' 'Gegharkunik province' 'Karki'
 'Nakhichevan Autonomous Republic' 'Yukhari Askipara' 'Barkhudarli'
 'Qazakh Rayon' 'Yaradullu' 'Khuriya Muriya Islands' 'Korean Peninsula'
 'plazas de soberanía' 'Military Demarcation Line' 'Sakhalin/Karafuto'
 'Kuril/Chishima, and South Kuril/Chishima Islands' 'Dokdo/Takeshima'
 'Heglig' 'Naf River' 'Noktundo' 'Pedra Branca' 'Si

- 48,014 rows.
- "claimant" contains 731 distinct sources in NL. Some claimants are series, e.g. comma-seperated.
- 122 distinct verdicts in "factcheck_verdict" column.
- Some claimants are generic, e.g. "Facebook Post".
- Contains fact-checking entity in "claim_source".
    - The Borderlines subset is weird. Its claimants and labels are geographic locations
    - The CheckYourFact subset contains a small amount of claimants, many of which are generic. Its labels are binary.
    - The FactCheckNI subset is more traditional, but is Ireland-centric. Binary labels.
    - The Factly subset is India-based and contains few unique properly formatted claimants. Labels are ternary, including Half True.
    - The PolitiFact subset is traditional with the same ternary labels.
    - The ClimateFeedback subset contains a small number of claimants. All are labeled False.
    - The HealthFeedback subset contains a small number of claimants. All are labeled False or Half True.
    - The ScienceFeedback subset is traditional. Ternary labels.
    - The SriLankaFactCrescendo subset has mainly generic claimants. All are labeled False.
- `Preliminary thoughts`: Keep PolitiFact and ScienceFeedback subsets.


In [27]:
chosen_prefixes = ["sciencefeedback", "politifact"]

# This slightly ugly thing just picks out the chosen subset
druid_plus_subset_df = druid_plus_df[
    np.any(
        np.array(
            [
                druid_plus_df["claim_id"].str.startswith(prefix)
                for prefix in chosen_prefixes
            ]
        ),
        axis=0,
    )
]
druid_plus_subset = datasets.Dataset.from_pandas(druid_plus_subset_df)
druid_plus_subset_df

,id,claim_id,claim_source,claim,claimant,claim_date,evidence_source,evidence,evidence_date,factcheck_verdict,is_gold,relevant,evidence_stance
28984,politifact_1,politifact_1,https://www.politifact.com/factchecks/2024/aug...,"Former President Donald Trump ""intends to enac...",Kamala Harris,2024-08-22,https://www.politifact.com/factchecks/2024/aug...,"During his 2024 presidential campaign, former ...",2024-08-22,Half True,True,True,insufficient-supports
28985,politifact_1_ret_b10_gn,politifact_1,https://www.politifact.com/factchecks/2024/aug...,"Former President Donald Trump ""intends to enac...",Kamala Harris,2024-08-22,https://www.politifact.com/article/2021/jan/19...,"• That said, Trump did implement several key p...",2021-01-19,Half True,False,True,insufficient-neutral
28986,politifact_1_ret_b12_gn,politifact_1,https://www.politifact.com/factchecks/2024/aug...,"Former President Donald Trump ""intends to enac...",Kamala Harris,2024-08-22,https://rollcall.com/2024/08/23/fact-checking-...,"She’s referring to Trump’s plan to impose ""uni...",2024-08-23,Half True,False,None,None
28987,politifact_1_ret_b13_gn,politifact_1,https://www.politifact.com/factchecks/2024/aug...,"Former President Donald Trump ""intends to enac...",Kamala Harris,2024-08-22,https://www.dallasnews.com/news/politifact/202...,Harris is the second woman to become a major-p...,2024-08-22,Half True,False,None,None
28988,politifact_1_ret_b14_gn,politifact_1,https://www.politifact.com/factchecks/2024/aug...,"Former President Donald Trump ""intends to enac...",Kamala Harris,2024-08-22,https://www.politifact.com/factchecks/list/?pa...,Stand up for the facts! Our only agenda is to ...,2024-09-10,Half True,False,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
43916,sciencefeedback_97_ret_bn_g0,sciencefeedback_97,https://science.feedback.org/review/altered-im...,“Governor of Maine orders restaurant staff to ...,"Sherri Tenpenny, Paul Joseph Watson",2020-08-18,https://science.feedback.org/review/altered-im...,Restaurant staff in Maine are not required to ...,2020-08-27,False,False,None,None
43917,sciencefeedback_97_ret_bn_g1,sciencefeedback_97,https://science.feedback.org/review/altered-im...,“Governor of Maine orders restaurant staff to ...,"Sherri Tenpenny, Paul Joseph Watson",2020-08-18,https://tonylutz.com/index.php/539-dog-cone-co...,The Governor of Maine has ordered restaurant s...,2020-08-19,False,False,None,None
43918,sciencefeedback_97_ret_bn_g12,sciencefeedback_97,https://science.feedback.org/review/altered-im...,“Governor of Maine orders restaurant staff to ...,"Sherri Tenpenny, Paul Joseph Watson",2020-08-18,https://acadiaonmymind.com/2020/07/acadia-nati...,"Gov. Janet Mills, who extended the state of ci...",2020-09-05,False,False,None,None
43919,sciencefeedback_97_ret_bn_g13,sciencefeedback_97,https://science.feedback.org/review/altered-im...,“Governor of Maine orders restaurant staff to ...,"Sherri Tenpenny, Paul Joseph Watson",2020-08-18,https://www.businessinsider.com/coronavirus-th...,A state emergency order issued by Gov. David I...,2020-07-12,False,False,None,None


- Resulting subset has 14,654 rows, 310 unique claimants and ternary labels.

## Emergent.Info

In [28]:
emergent = datasets.load_dataset(
    "csv", data_files=dogpath("emergent.csv"), split="train"
)
emergent

ValueError: File 'emergent_phase1_clean_2018_7_2/emergent_phase1_clean_2018_7_2.csv' is not in the registry.

- Only 187 rows
- "fact_tag_phase1" in {false, true, Unverified}
- All rows are dated Jul 3 or missing a date
- The sources only appear as local URLs to articles.
- `Preliminary thoughts`: Skip this dataset. It's too small and hard to work with.

## FakeHealth

In [29]:
fakehealth = datasets.load_dataset(
    "json",
    data_files=[dogpath("health_release.json"), dogpath("health_story.json")],
    split="train",
)
fakehealth

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['link', 'title', 'description', 'original_title', 'rating', 'reviewers', 'category', 'tags', 'source_link', 'summary', 'criteria', 'news_id', 'news_source'],
    num_rows: 2296
})

- 2,296 rows.
- "title"s of the fact-checks and "original_title". Ratings in [0,5]. "Reviewers" column. "source_link". Contains a "news_source" column with the name of the institution e.g. Hospital performing the health-check.

## FakeNewsNet

In [30]:
fakenewsnet_fake = datasets.load_dataset(
    "csv",
    data_files=dogpath("politifact_fact_check_fake.csv"),
    split="train",
)
fakenewsnet_fake = add_binary_label_column(fakenewsnet_fake, False)

fakenewsnet_real = datasets.load_dataset(
    "csv",
    data_files=dogpath("politifact_fact_check_real.csv"),
    split="train",
)
fakenewsnet_real = add_binary_label_column(fakenewsnet_real, True)

fakenewsnet = datasets.concatenate_datasets(
    [fakenewsnet_fake, fakenewsnet_real]
)
fakenewsnet

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['id', 'news_url', 'title', 'tweet_ids', 'label'],
    num_rows: 1056
})

- 1056 rows
- From PolitiFact
- A few duplicates
- "news_url" of stories (not cleaned). This is the only source information. 
- Labels in real/fake (I've keyed them to booleans)

## GossipCop

In [31]:
gossipcop_fake = datasets.load_dataset(
    "csv",
    data_files=dogpath("gossipcop_fact_check_fake.csv"),
    split="train",
)
gossipcop_fake = gossipcop_fake.add_column(
    "label", [False] * len(gossipcop_fake)
)

gossipcop_real = datasets.load_dataset(
    "csv",
    data_files=dogpath("gossipcop_fact_check_real.csv"),
    split="train",
)
gossipcop_real = gossipcop_real.add_column(
    "label", [True] * len(gossipcop_real)
)

gossipcop = datasets.concatenate_datasets([gossipcop_fake, gossipcop_real])
gossipcop

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['id', 'news_url', 'title', 'tweet_ids', 'label'],
    num_rows: 22140
})

- Same format as previous. Concerns celebrity news.

## Liar-PLUS

In [32]:
liar_plus = datasets.load_dataset(
    "csv",
    data_files=[
        dogpath("liar_plus_test.tsv"),
        dogpath("liar_plus_train.tsv"),
        dogpath("liar_plus_val.tsv"),
    ],
    split="train",
    delimiter="\t",
    column_names=[
        "ID",
        "Label",
        "Statement",
        "Subjects",
        "Speaker",
        "Speaker's Job Title",
        "State",
        "Party Affiliation",
        # These are cumulative label sums on the speaker including the current
        # statement
        "Barely True Sum",
        "False Sum",
        "Half-true Sum",
        "Mostly True Sum",
        "Pants on Fire Sum",
        "Context",  # Location or context of statement
        "Verdict Justification",
    ],
)
print(set(liar_plus["Party Affiliation"]))
liar_plus

Generating train split: 0 examples [00:00, ? examples/s]

{'county-commissioner', 'libertarian', 'organization', None, 'liberal-party-canada', 'independent', 'talk-show-host', 'democrat', 'republican', 'Moderate', 'democratic-farmer-labor', 'business-leader', 'newsmaker', 'journalist', 'education-official', 'government-body', 'ocean-state-tea-party-action', 'labor-leader', 'constitution-party', 'activist', 'tea-party-member', 'columnist', 'state-official', 'none', 'green'}


Dataset({
    features: ['ID', 'Label', 'Statement', 'Subjects', 'Speaker', "Speaker's Job Title", 'State', 'Party Affiliation', 'Barely True Sum', 'False Sum', 'Half-true Sum', 'Mostly True Sum', 'Pants on Fire Sum', 'Context', 'Verdict Justification', '__index_level_0__'],
    num_rows: 12793
})

- 12,793 rows.
- "Speaker" are formatted lowercase, dash-seperated; as are labels.
- Contain 24 distinct "Party Affiliation" values in {'republican', 'government-body', 'organization', 'columnist', 'none', 'tea-party-member', None, 'activist', 'constitution-party', 'democratic-farmer-labor', 'business-leader', 'labor-leader', 'liberal-party-canada', 'ocean-state-tea-party-action', 'libertarian', 'county-commissioner', 'education-official', 'democrat', 'independent', 'green', 'newsmaker', 'Moderate', 'talk-show-host', 'state-official', 'journalist'}
- Labels in the PolitiFact Truth-o-meter.

## MisInfo General

In [33]:
# We have to use snapshot_download to download the dataset together with
# the source metadata without using huggingface-cli.
misinfo_general_dir = snapshot_download(
    repo_id="ioverho/misinfo-general",
    repo_type="dataset",
    token=HF_HUB_TOKEN,
)
misinfo_general_dir

Fetching 40 files:   0%|          | 0/40 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/20.3k [00:00<?, ?B/s]

c:\Users\tharg\anaconda3\envs\gnn_cf\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tharg\.cache\huggingface\hub\datasets--ioverho--misinfo-general. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


db_schema.png:   0%|          | 0.00/19.5k [00:00<?, ?B/s]

.gitattributes:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

...

2022-00006-of-00009.parquet:   0%|          | 0.00/245M [00:00<?, ?B/s]

2022-00007-of-00009.parquet:   0%|          | 0.00/247M [00:00<?, ?B/s]

2022-00008-of-00009.parquet:   0%|          | 0.00/242M [00:00<?, ?B/s]

metadata.db:   0%|          | 0.00/769M [00:00<?, ?B/s]

OSError: [Errno 28] No space left on device

In [34]:
# The Source metadata is stored in a duckdb database - see
# https://huggingface.co/datasets/ioverho/misinfo-general
metadata_db = duckdb.connect(
    os.path.join(misinfo_general_dir, "metadata.db"),
    read_only=True,
)
print(
    metadata_db.sql(
        """
    SHOW TABLES
    """
    )
)

NameError: name 'misinfo_general_dir' is not defined

In [ ]:
for table_name in ["articles", "covid_articles", "sources", "topics"]:
    print(table_name.capitalize())
    print(
        metadata_db.sql(
            f"""
      DESCRIBE {table_name}
      """
        )
    )

Articles
┌──────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name    │ column_type │  null   │   key   │ default │  extra  │
│     varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ article_id       │ VARCHAR     │ NO      │ PRI     │ NULL    │ NULL    │
│ year             │ USMALLINT   │ YES     │ NULL    │ NULL    │ NULL    │
│ source           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ title            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ publication_date │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ topic_id         │ UINTEGER    │ YES     │ NULL    │ NULL    │ NULL    │
│ url              │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ covid_article    │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
└───────────────

### MisInfo General: Sources

In [ ]:
metadata_db.sql(
    f"""
    SELECT *
    FROM sources
    LIMIT 50
    """
)

┌───────────────────┬────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────┬────────────────┬─────────────────────────┬───────────────────────────────────┬───────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────┬────────────┐
│      source       │                              url                               │                                                                      label                                                                       │                         

In [ ]:
# Save the source table as HuggingFace Dataset
# We just want some one representative URL for each source to strip the domain.
misinfo_general_sources = datasets.Dataset.from_pandas(
    metadata_db.sql(
        f"""
    WITH sources_with_representative_urls
    AS (
        SELECT source, MIN(url) representative_article_url
        FROM articles
        GROUP BY source
    )
    SELECT *
    FROM (sources_with_representative_urls NATURAL JOIN sources)
    """
    ).df()
)
misinfo_general_sources

Dataset({
    features: ['source', 'representative_article_url', 'url', 'label', 'bias', 'factuality', 'country', 'type', 'popularity', 'credibility', 'reasoning', 'conspiracy', 'pseudosci', 'check_date'],
    num_rows: 474
})

In [ ]:
# Investigate the few articles that have one of 'pseudosci' and 'conspiracy'
# but not both
metadata_db.sql(
    f"""
   SELECT *
   FROM sources
   WHERE (conspiracy IS NULL AND NOT pseudosci IS NULL) OR (NOT conspiracy IS NULL AND pseudosci IS NULL)
   """
)

┌───────────────────────────┬─────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────┬─────────┬──────────┬───────────────────────────────────┬───────────────────────────────┬───────────┬────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────┬────────────┐
│          source           │                             url                             │                                                                      label                                                                       │                                                 bias                                                  │         

- Only 488 distinct sources/rows.
- "source" are in NL, lowercase collapsed together with no spaces - <u>however</u>, these don't seem to map directly to domains. For example, the dataset card contains the source "themanchestereveningnews" whose associated domain cuts the "the"-prefix.
- "url" for the FC article.
- "label" in {'left', 'left-center', 'least biased', 'right-center', 'right', 'pro-science', 'satire', 'questionable source', 'conspiracy-pseudoscience'}. These are essentially high-level labels of the source. 34% are Conspiracy-Pseudoscience, 19% are Questionable Source.
- "bias" in {'extreme left', 'left', 'left-center', 'least biased', 'right-center', 'right', 'extreme right'}. Missing from 22%. Otherwise top values are Right (0.21), Left-Center (0.17) and Extreme Right (0.15).
- "factuality" in {'very high', 'high', 'mostly factual', 'mixed', 'low', 'very low'}. Missing from 23%. Top are Low (0.28), Mixed (0.26), High (0.23).
- "country".
- "type" describing the medium, e.g. Website, Newspaper, TV Station... Missing from 5%. Top are Website (0.65), Newspaper (0.12) and Organization/Foundation (0.07). 
- "popularity" in {minimal, medium, high} - <u>can possibly be used for ranking sources!</u>
- "credibility" in {high (0.32), medium (0.1), low (0.45)}.
- "reasoning" column with tags for the reasonings of the label - only for 14% of the data.
- "conspiracy" in {'weak', 'mild', 'moderate', 'strong', 'tin foil hat'} (or NULL) and "pseudosci" in {'weak', 'mild', 'moderate', 'strong', 'quackery'} (or NULL), specifically for sources labeled as Conspiracy-Pseudoscience. Almost but not always coming in pairs (32% have "conspiracy" value, 33% have "pseudosci" value), i.e. seem to always come in pairs. Among tagged datapoints, the labels skew strong. The five situations that do not have pairwise values have "pseudoscience" labels but no "conspiracy" labels.
- "check_date"s which contain clock-time but always at 00:00:00.

## MM_COVID19

In [35]:
mm_covid19 = datasets.load_dataset(
    "csv", data_files=dogpath("mm_covid19.csv"), split="train"
)
mm_covid19

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['news_id', 'agency', 'claim', 'fact_url', 'label', 'lang', 'ref_source', 'ref_source_url', 'statement'],
    num_rows: 7423
})

In [36]:
# Data Wrangler seems to struggle with this dataset (possible formatting error?), so let's use Pandas.
mm_covid19_df = mm_covid19.to_pandas()
mm_covid19_df[mm_covid19_df["label"] == "fake"]

,news_id,agency,claim,fact_url,label,lang,ref_source,ref_source_url,statement
85,politifact-7886690361,politifact,no one has died from the coronavirus in the Un...,https://www.politifact.com/factchecks/2020/mar...,fake,en,None,https://www.instagram.com/p/B9N9_4WFC_y/?utm_s...,no one has died from the coronavirus in the Un...
86,snopes-5121448257,snopes,Parents can receive a case of baby formula dur...,https://www.snopes.com/fact-check/free-case-of...,fake,en,None,NONE,free baby formula covid-19
87,snopes-8082605617,snopes,Drinking boiled garlic water will cure COVID-19.,https://www.snopes.com/fact-check/garlic-cure-...,fake,en,None,NONE,Garlic water covid-19
88,snopes-6798306093,snopes,A photograph of non-respirator ear loop mask p...,https://www.snopes.com/fact-check/cloth-masks-...,fake,en,None,NONE,Non-Respirator masks ineffective
89,snopes-41249603,snopes,When U.S. President Donald Trump was asked wha...,https://www.snopes.com/fact-check/trump-covid-...,fake,en,None,NONE,trump covid-19 deaths hurt anyone
...,...,...,...,...,...,...,...,...,...
3942,reuters-9037325584,reuters,"in china the coronavirus only hit wuhan , chin...",https://www.reuters.com/article/uk-factcheck-c...,fake,en,"{""_id"":{""$oid"":""5efc1aa766a57354dee6135a""},""id...",https://www.facebook.com/photo.php?fbid=363702...,"in china the coronavirus only hit wuhan , chin..."
3943,reuters-4257791404,reuters,steam therapy kills coronavirus,https://www.reuters.com/article/uk-factcheck-s...,fake,en,"{""_id"":{""$oid"":""5efe3d7266a57354deea2999""},""id...",https://www.siasat.com/how-kill-coronavirus-le...,Steam therapy kills coronavirus
3944,snopes-7413781860,snopes,After discovering a doomsday asteroid was appr...,https://www.snopes.com/fact-check/coronavirus-...,fake,en,"{""_id"":{""$oid"":""5efc1aad66a57354dee61368""},""id...",https://www.snopes.com/uploads/2020/03/faceboo...,world leaders created the COVID-19 pandemic di...
4030,PolitiFact-9390000809,PolitiFact,The flu didn’t kill any Americans this year.,https://www.politifact.com/factchecks/2020/may...,fake,en,"{""screen_name"":[""profile.php"",""peppicole"",""bar...",https://www.facebook.com/john.formato/posts/10...,americans flu died covid covid


In [37]:
# Let's drop the non-statements
mm_covid19_df.dropna(subset=["claim"], inplace=True)

In [38]:
def _augment_with_canonical_urls_and_authors(row):
    canonical_link = None
    authors = None
    if row["ref_source"]:
        ref_sources_dict = json.loads(row["ref_source"])
        # Some links consist only of whitespace.
        if "canonical_link" in ref_sources_dict.keys() and len(
            ref_sources_dict["canonical_link"].strip()
        ):
            canonical_link = ref_sources_dict["canonical_link"]
        # We leave empty author lists as None for easier dropping in Pandas in
        # the following.
        if "authors" in ref_sources_dict.keys() and len(
            ref_sources_dict["authors"]
        ):
            authors = ref_sources_dict["authors"]
    row["canonical_link"] = canonical_link
    row["authors"] = authors
    return row


mm_covid19 = mm_covid19.map(_augment_with_canonical_urls_and_authors)

Map:   0%|          | 0/7423 [00:00<?, ? examples/s]

In [39]:
mm_covid19_df = mm_covid19.to_pandas()
# Drop rows with neither authors nor canonical URLs
mm_covid19_df.dropna(
    subset=["canonical_link", "authors"], how="all", inplace=True
)
mm_covid19_df[mm_covid19_df["label"] == "fake"]

,news_id,agency,claim,fact_url,label,lang,ref_source,ref_source_url,statement,canonical_link,authors
887,AAPFactCheck-3550501923,AAP FactCheck,A post claims a COVID-19 vaccine has already ...,https://www.aap.com.au/covid-19-vaccine-whistl...,fake,en,"{""_id"":{""$oid"":""5f32b283dfd2b7a505e30047""},""id...",https://www.standard.co.uk/lifestyle/london-li...,sounds lot whistleblower like study informatio...,https://www.standard.co.uk/lifestyle/london-li...,[Susannah Butter]
890,FactCheck.org-5735755418,FactCheck.org,A Colorado bill is “removing exemptions” for ...,https://www.factcheck.org/2020/06/colorado-vac...,fake,en,"{""_id"":{""$oid"":""5f32b53ddfd2b7a505e301ab""},""id...",https://www.cdc.gov/coronavirus/2019-ncov/case...,nonmedical includes exemptions colorado bill v...,https://www.cdc.gov/coronavirus/2019-ncov/case...,None
896,Newsmeter.in-106620864,Newsmeter.in,Dawood Ibrahim died of Covid-19.,https://newsmeter.in/claim-that-dawood-ibrahim...,fake,en,"{""_id"":{""$oid"":""5f32b5a4dfd2b7a505e301ef""},""id...",https://t.co/eGVK1vKNpe,ibrahim dawood died covid covid,https://twitter.com/sadhvi_pandey/status/12691...,None
903,PolitiFact-8837527317,PolitiFact,Italy “went against the WHO’s ‘law’” against ...,https://www.politifact.com/factchecks/2020/may...,fake,en,"{""_id"":{""$oid"":""5f32b6d5dfd2b7a505e302a2""},""id...",https://www.axios.com/italy-coronavirus-cases-...,bacterium virus covid covid,https://www.axios.com/italy-coronavirus-cases-...,"[Https, Www.Axios.Com Authors Newsdesk]"
904,PolitiFact-3176777157,PolitiFact,Says the state “health secretary says don’t g...,https://www.politifact.com/factchecks/2020/may...,fake,en,"{""_id"":{""$oid"":""5f32b6e8dfd2b7a505e302b4""},""id...",https://twitter.com/TomTiffanyWI/status/126524...,wisconsini tiffany officials covid,https://twitter.com/tomtiffanywi/status/126524...,None
...,...,...,...,...,...,...,...,...,...,...,...
3939,reuters-5453619223,reuters,daffodils at welsh castle honour covid-19 victims,https://www.reuters.com/article/uk-factcheck-d...,fake,en,"{""_id"":{""$oid"":""5efe3d6066a57354deea2904""},""id...",https://www.facebook.com/WalesforaUnitedKingdo...,daffodils at welsh castle honour covid-19 victims,https://www.facebook.com/WalesforaUnitedKingdo...,None
3940,reuters-6054450795,reuters,video shows healthcare workers celebrating fra...,https://www.reuters.com/article/uk-factcheck-f...,fake,en,"{""_id"":{""$oid"":""5efc1a9966a57354dee6133d""},""id...",https://www.facebook.com/salone4all/videos/vb....,France has discovered the vaccine for Corona v...,https://www.facebook.com/salone4all/videos/bre...,None
3941,reuters-6054554103,reuters,mandatory workplace leave and school closures ...,https://www.reuters.com/article/uk-factcheck-c...,fake,en,"{""_id"":{""$oid"":""5efc1a9c66a57354dee61345""},""id...",https://www.facebook.com/photo.php?fbid=102157...,closure slow Covid-19,https://www.facebook.com/oshea.bunkin/posts/10...,None
3942,reuters-9037325584,reuters,"in china the coronavirus only hit wuhan , chin...",https://www.reuters.com/article/uk-factcheck-c...,fake,en,"{""_id"":{""$oid"":""5efc1aa766a57354dee6135a""},""id...",https://www.facebook.com/photo.php?fbid=363702...,"in china the coronavirus only hit wuhan , chin...",https://www.facebook.com/kingdomepiccentre/pos...,None


In [40]:
mm_covid19 = datasets.Dataset.from_pandas(mm_covid19_df)

mm_covid19_authors = extract_set_of_authors_from_list_column(
    mm_covid19, "authors"
)
print(len(mm_covid19_authors))
mm_covid19_authors

400


{'- Micki Jackson',
 'Aap Factcheck',
 'Aaron C. Davis',
 'Aaron Kandola',
 'Aaron Miller',
 'Abc News',
 'Abeahm Al.Com',
 'Abigail Weinberg',
 'Acharya Narendra Dev College',
 'Aiswarya Sriram',
 'Akhilesh Mishra',
 'Alex Bell',
 'Alex Kasprak',
 'Alexa Mae Asperin',
 'Ali Eslami',
 'Alice Dragoon',
 'Alternate Author',
 'Amanda Barrell',
 'Amee Vanderpool',
 'Amritha Mohan',
 'Amy Smith',
 'Ana Sandoiu',
 'Andrew Hall',
 'Andrew Joseph',
 'Angela Chen',
 'Angela Dewan',
 'Angelo Fichera',
 'Ann Pietrangelo',
 'Anna Beahm',
 'Anna Merlan',
 'Anna Smith',
 "Anne D'Innocenzio",
 'Anne Flaherty',
 'Anne Trafton',
 'Anonymous Said',
 'Antonio Regalado',
 'Antwort An',
 'Anusha Puppala',
 'Ap News',
 'April',
 'Around The World To Do Stories For Cbc',
 'Art Spectrum',
 'Artemisia Annua',
 'Arun Kumar',
 'As My Human Makeup Consists Of Having A Very Versatile Life With Lots Of Twists',
 'Ashish Bhosale',
 'Ashutosh Kumar Pandey',
 'Associated Press',
 'Author',
 'Aylin Woodward',
 'Baffour

- 7423 rows
- Covid19-related
- 46% missing claims which are tweets of statements or nonverifiable claims, probably for dataset balance.
- Fact-check URLs, fact-checking organization in "agency"
- Labels in {real, fake}
- The direct URL to the claim can be found in "ref_source_url", but this is often archive entries or URL-shortenings. The canonical URL together with an optional list of authors can be extracted from a nested JSON object in "ref_source".

## MultiFC

In [41]:
multifc = datasets.load_dataset(
    "csv",
    data_files=[
        dogpath("multifc_public_data_dev.tsv"),
        dogpath("multifc_public_data_train.tsv"),  # The test set does not
        # contain labels
    ],
    delimiter="\t",
    split="train",
    column_names=[
        "claimid",
        "claim",
        "label",
        "claim url",
        "reason",
        "category",
        "speaker",
        "checker",
        "tags",
        "article title",
        "publish date",
        "claim date",
        "claim entities",
    ],  # Derived from the paper, but their exact ordering was manually
    # achieved
)
multifc

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['claimid', 'claim', 'label', 'claim url', 'reason', 'category', 'speaker', 'checker', 'tags', 'article title', 'publish date', 'claim date', 'claim entities'],
    num_rows: 31355
})

In [42]:
if RUN_ALL:
    print(
        f"Claim dates: {retrieve_heterogeneously_formatted_years(multifc['claim date'])}"
    )
    print(
        f"Publish dates: {retrieve_heterogeneously_formatted_years(multifc['publish date'])}"
    )

Claim dates: ['1995', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2025']
Publish dates: ['1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2025']


- Nearly 35,000 rows
- Heterogenous dataset; some columns have differently formatted data, including the "Claim URL" and "Publish Date" columns
- Claims, heterogenous labels (3590 distinct), URLs, textual label reason, "Speaker" row, "Checker" row (only 354 distinct), titles
- Many missing elements, some duplicates

## NewsMediaReliability

In [43]:
news_media_reliability_emnlp = datasets.load_dataset(
    "csv",
    data_files=dogpath("news_media_reliability_emnlp"),
    delimiter="\t",
    split="train",
)

news_media_reliability_emnlp

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['source_url', 'source_url_normalized', 'ref', 'fact', 'bias'],
    num_rows: 1066
})

- URLs, factuality in {low 0.24, mixed 0.25, high 0.51}, bias in {(extreme) (center)-{left, right}}
- 1066
- All sources are from MBFC

In [44]:
news_media_reliability_acl = datasets.load_dataset(
    "csv",
    data_files=dogpath("news_media_reliability_acl"),
    delimiter="\t",
    split="train",
)

news_media_reliability_acl

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['source_url', 'source_url_normalized', 'ref', 'fact', 'bias'],
    num_rows: 859
})

- A cleansed version of the EMNLP dataset. The authors argue that the center-* labels are inconsistent and therefore the given sources were removed. Furthermore, the extreme labels are collapsed into the regular ones. 
- Hence has bias in {right 0.41, center 0.32, left 0.28} and factuality in {high 0.53, mixed 0.29, low 0.19}
- 859 rows
- One missing reference

Preliminary thoughts: Keep the EMNLP dataset for now. It's a superset, and the extra labels are chosen by MBFC.

## PolitiFact Fact Check Dataset

In [45]:
politifact_fact_check = datasets.load_dataset(
    "json",
    data_files=politifact_fact_check_path,
    split="train",
)
politifact_fact_check

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['verdict', 'statement_originator', 'statement', 'statement_date', 'statement_source', 'factchecker', 'factcheck_date', 'factcheck_analysis_link'],
    num_rows: 21152
})

In [46]:
if RUN_ALL:
    print(
        f"Statement years: {retrieve_heterogeneously_formatted_years((politifact_fact_check['statement_date']))}"
    )
    print(
        f"Fact-check years: {retrieve_heterogeneously_formatted_years((politifact_fact_check['factcheck_date']))}"
    )

Statement years: ['2000', '2002', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022']
Fact-check years: ['2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022']


- 21,152 rows.
- Uses the PolitiFact Truth-o-meter.
- "statement_originator" in NL; 4565 distinct.
- "statement_source", being source medium or context, such as speech, campaign, news, etc. (13 distinct).
- "factchecker" (NL).
- "factcheck_analysis_link", i.e. the fact check URL.
- Skews toward negative labels, i.e. lies.
- Dated to a broad timespan in the 2000s.

## ReCoVery

In [47]:
recovery = datasets.load_dataset(
    "csv", data_files=dogpath("recovery.csv"), split="train"
)
recovery

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['Unnamed: 0', 'news_id', 'url', 'publisher', 'publish_date', 'author', 'title', 'image', 'body_text', 'political_bias', 'country', 'reliability'],
    num_rows: 2029
})

In [48]:
recovery_authors = extract_set_of_authors_from_list_column(recovery, "author")
print(len(recovery_authors))
recovery_authors

1086


{'Francesco Canepa',
 'Amanda Mull',
 'Natalie Chyi',
 'Irene Jiang',
 'Dennis J. Snower',
 'Bryan Alexander',
 'Mitchell Nathanson',
 'Ben Winck',
 'Kyle Rempfermy',
 'Bonnie Berkowitz',
 'John Vibes',
 'Zack Stanton',
 'Danielle Garrand',
 'Megan Cerullo',
 'J.B. Shurk',
 'Joseph Puder',
 'Lloyd Billingsley',
 'Caitlin Emma',
 'Phillip Schneider',
 'Anthony L. Fisher',
 'Henry Grabar',
 'Dan Mangan',
 'Callaghan OHare',
 'Alex Nicoll',
 'Irene Wang',
 'Sarah Ewall-Wice',
 'Joseph Zeballos-Roig',
 'Kris Malysz',
 'Phil Galewitz',
 'Carl Zimmer',
 'Alexandria White',
 'Scott Hensley',
 'Keith Birmingham',
 'Christopher Schaberg',
 'Jerry Adler',
 'Barbara Booth',
 'Ramy Inocencio',
 'Jesse Chase-Lubitz',
 'Tovia Smith',
 'Dylan Bouscher',
 'Michelle R. Smith',
 'Kelly Mclaughlin',
 'Lisa M. Krieger',
 'Matt Agorist',
 'Jonathan Lai',
 'Maggie Shannon',
 'Jon Gaunt',
 'Brandon Sudge',
 'Jennifer Liu',
 'Deepa Bharath',
 'Harriet Blair Rowan',
 'Stephanie Zimmermann',
 'Ryan Carter',
 'K

- 2029 rows
- Unnamed column which seems to be equivalent to the "news_id" column.
- "url", "publisher" in NT, "author" column with array values, often empty, "title", political_bias in (Extreme) (Center-) {left, right}, binary reliability score
- Labels are derived from MBFC and NewsGuard. From the paper: "A news site is reliable if its NewsGuard score is greater than 90, and its factual reporting on MBFC is very high or high. A news site is unreliable if its NewsGuard score is less than 30, and its factual reporting on MBFC is below mixed".
- Hence <u>this is an extreme dataset</u>; middling values do not appear.
- Articles are (all?) published in 2020
- 82% of articles from the US, 10% from Russia
- Some missing values and duplicates

## Snopes

In [49]:
snopes = datasets.load_dataset(
    "csv", data_files=dogpath("snopes.csv"), split="train"
)
snopes

ValueError: File 'snopes_phase1_clean_2018_7_3/snopes_phase1_clean_2018_7_3.csv' is not in the registry.

In [ ]:
set(snopes["fact_rating_phase1"])

{'correct attribution',
 'false',
 'legend',
 'misattributed',
 'miscaptioned',
 'mixture',
 'mostly false',
 'mostly true',
 'outdated',
 'scam',
 'true',
 'unproven'}

In [ ]:
if RUN_ALL:
    retrieve_heterogeneously_formatted_years(snopes["article_date_phase1"])

- 37,957 rows of which 5,640 are unique.
- Again, URLs are to sources used and/or claims on e.g. Twitter, hence this is probably unusable.

# Comparing/merging datasets

The following table is sorted in descending order by the following key: Distinct sources, lists of authors, domain URLs, article URLs, other.

| Name | Num. Distinct sources | Origin(s) | Source designation | Labels | Other useful attributes | Date^ | Dirty?~ | Comments |
| ---| --- | --- | --- | ---| --- | --- | --- | --- |
| Domain Ratings | 11,520 | NewsGuard, MBFC, Ad Fontes Media, crowdsourcing, 60 professional fact-checkers, the Iffy index | Domain | Multiple kinds of scores, mostly in [0, 1] | Statistically derived scores, chiefly their custom PC1 score | Dataset: 2023 | False | The Iffy score is 1 if a domain does not appear on the Iffy index, else 0, hence it can have "false positives" |
| PolitiFact Fact Check | 4565 | PolitiFact | NL | Truth-o-meter | Fact-checker | Data: Statements from 2000 + 2002 + 2007-2022, FCs from 2007-2022 | True | Labels skew negative. |
| MultiFC | 32,628 URLs, 3997 Speakers | Heterogenous | Claim URL, Speaker (NL)| Heterogenous (116 distinct) || Data: Claims from 1995 + 2007-2018, Fact Checks from 1995-2018 | True | Very dirty and heterogeneous |
| Liar-PLUS | 3309 | PolitiFact | Source in normalized lower-case dash-seperated NL | Truth-o-meter | Speaker's job title, cumulative sums of scores per source | Dataset: 2018 | True ||
| AVeriTeC| 1571 speakers, 701 reporting sources | Heterogenous | Speaker (NL) + Reporting source (NL) + URL | {Refuted, Conflicting Evidence/Cherrypicking, Supported, Not Enough Evidence} | | Data: 1919-1922 + 2008-2009 + 2011-2020 | False | Each element may have any combination of speaker, reporting source and URL |
| NewsMediaReliability-EMNLP| 1066 | MBFC | (Normalized) Domain | {low, mixed, high} | bias (MBFC style) | Dataset: 2019| False | |
| NewsMediaReliability-ACL| 859| MBFC | (Normalized) Domain  | {low, mixed, high} | bias (MBFC style, extremes collapsed) | Dataset: 2021| True| Cleansed subset of EMNLP |
| MisInfo General: Sources | 488 | MBFC | Lowercase no-space NL | Factuality in {'very high', 'high', 'mostly factual', 'mixed', 'low', 'very low'}, Credibility in {high (0.32), medium (0.1), low (0.45)} | MBFC style bias, high level "label" of source, popularity, reasoning (of label), conspiracy and pseudosci | FCs in 2023-2024 | True | Source names map often but not always to URLs |
| FakeHealth | 462 sources, 2295 URLs | HealthNewsReview.org | NL + Domain | [0, 5] (higher better) | (Not) Satisfactory answer JSON | Dataset: 2020 | False | Health-related news |
| DRUID+ (subset) | 310 sources | PolitiFact, Science Feedback | NL | {False, Half True, True} || Data: 2013-2024 | False ||
| Breaking! Fake News Dataset | 209| Own| Author (NL, name or org) + Domain | {questionable, fake} || Dataset: 2019| True | Negative dataset, only bad sources |
| ReCoVery | 55 publishers, ~1086 authors | MBFC and NewsGuard | Publishers in NL, URLs, arrays of authors in NL | {0, 1} mapped from NewsGuard and MBFC | MBFC style bias | Data: 2020 | True | Concerns the Covid pandemic. Contains only datapoints with extreme labels: From the paper: "A news site is reliable if its NewsGuard score is greater than 90, and its factual reporting on MBFC is very high or high. A news site is unreliable if its NewsGuard score is less than 30, and its factual reporting on MBFC is below mixed". |
| MM-COVID19 | 1845 URLs, ~400 authors | 38 distinct | Direct URLs and lists of authors in NL | {real, fake} || Dataset: 2020 | True | Covid-related. Fairly dirty.|
| FakeNewsNet | 967 URLs | PolitiFact | URL | Binary || Dataset: 2017 | True ||
| Emergent.Info | 165 URLs | Emergent.Info | URL | {false, true, Unverified} || Dataset: 2018 | True ||
| PolitiFact | ?< 73,766 URLs | PolitiFact | Only appear in NL in FC articles | Truth-o-meter || Data: 2007-2018 | False | Sources would have to be extracted from NL. |
| Snopes | ?< 37,958 URLs | Snopes | Only appear in NL in FC articles | {'correct attribution', 'false', 'legend', 'misattributed', 'miscaptioned', 'mixture', 'mostly false', 'mostly true', 'outdated', 'scam', 'true', 'unproven'} || 1996-2018 | False | Similar to the PolitiFact dataset |
| GossipCop | 20,691 URLs | GossipCop | URL | Binary || Dataset: 2017 | True | Concerns celebrity news |
| Covid Fake News: News | ?< 4,505 URLs | Heterogeneous | URLs used in FCs are included, but they may concern the news story itself or a person sharing it | Binary | type in {post, article, NULL} | 2016, 2018-2020 | True | Very dirty. Sources are hard to retrieve from the URLs |
| Covid Fake News: Claims | None | Heterogeneous | None | Binary | Dataset: 2020 | False | No sources. Concerns the Covid pandemic. Unusable. |

~ Duplicates or missing key values  
^ Explicit from data when possible, else dataset publication date. Loosely formatted dates are parsed in a generic manner and hence may be incorrect.

### Attempting to extract and compare differently formatted sets of sources

## NL

In [50]:
# Let's see how nice these sources look with dashes replaced by space
def _replace_dashes_with_space_in_dataset(col_name, row):
    if row[col_name]:
        row[col_name] = row[col_name].replace("-", " ")
    return row


liar_plus = liar_plus.map(
    lambda row: _replace_dashes_with_space_in_dataset("Speaker", row)
)

liar_plus["Speaker"]

Map:   0%|          | 0/12793 [00:00<?, ? examples/s]

['rick perry',
 'katrina shankland',
 'donald trump',
 'rob cornilles',
 'state democratic party wisconsin',
 'brendan doherty',
 'stand children tennessee',
 'john boehner',
 'sean patrick maloney',
 'mike pence',
 'mike berlon',
 'bill flores',
 'viral image',
 'facebook posts',
 'chris abele',
 'rick santorum',
 'tom niehaus',
 'marsha blackburn',
 'hillary clinton',
 'sarah palin',
 'chris sgro',
 'marco rubio',
 'donald trump',
 'marijuana policy project',
 'republican party florida',
 'lindsey graham',
 'hillary clinton',
 'republican national committee republican',
 'mike pence',
 'cliff stearns',
 'cory gardner',
 'sarah palin',
 'barack obama',
 'democratic congressional campaign committee',
 'mamie locke',
 'herman cain',
 'rick scott',
 'jeanne shaheen',
 'dennis kucinich',
 'kevin brady',
 'mitt romney',
 'doonesbury',
 'donald trump',
 'drew springer',
 'michele bachmann',
 'robert menendez',
 'hillary clinton',
 'barack obama',
 'steve brown',
 'hillary clinton',
 'republ

### Sets

In [ ]:
def _num_unique_source_contributions(key: str, superdict: dict):
    """
    Count how many unique sources each dataset contributes with. This is not an optimal generic function for this purpose, but it works for this context.
    """
    assert key in superdict.keys()
    remaining_source_dicts = [
        source_dict for (name, source_dict) in superdict.items() if name != key
    ]
    sources_sans_target = set().union(*remaining_source_dicts)
    return len(superdict[key].difference(sources_sans_target))


def _print_num_unique_source_contributions_per_item(superdict):
    print("\nUnique sources contributed per dataset:")
    for dataset_name in superdict.keys():
        print(
            f"{dataset_name}: {_num_unique_source_contributions(dataset_name, superdict)}"
        )


# Enforce lowercase on all lists for completeness
politifact_fact_check_nl_sources = set(
    str_list_to_lowercase(politifact_fact_check["statement_originator"])
)

liar_plus_nl_sources = set(str_list_to_lowercase(liar_plus["Speaker"]))

averitec_nl_sources = set(
    str_list_to_lowercase(averitec["speaker"] + averitec["reporting_source"])
)

fakehealth_nl_sources = set(str_list_to_lowercase(fakehealth["news_source"]))

fake_news_corpus_nl_sources = set(
    str_list_to_lowercase(fake_news_corpus["authors"])
)
recovery_nl_sources = set(str_list_to_lowercase(recovery["publisher"])).union(
    {author.lower() for author in recovery_authors}
)
multifc_nl_sources = set(str_list_to_lowercase(multifc["speaker"]))

mm_covid19_sources = {author.lower() for author in mm_covid19_authors}

druid_plus_sources = set(str_list_to_lowercase(druid_plus_subset["claimant"]))

all_nl_source_sets = {
    "PolitiFact Fact Check": politifact_fact_check_nl_sources,
    "Liar-PLUS": liar_plus_nl_sources,
    "AVeriTeC": averitec_nl_sources,
    "FakeHealth": fakehealth_nl_sources,
    "Fake News Corpus": fake_news_corpus_nl_sources,
    "ReCoVery": recovery_nl_sources,
    "MultiFC": multifc_nl_sources,
    "MM-COVID19": mm_covid19_authors,
    "DRUID+": druid_plus_sources,
}

# Note that a few of these natural language sources are in fact domains,
# and in general, they are somewhat noisy, e.g. contains entries such as
# "-the telegraph and the daily star". Hence you should probably run
# further cleaning if you are going to concretely use this downstream.
print("NL sources:")
_print_num_unique_source_contributions_per_item(all_nl_source_sets)

nl_sources = set().union(*all_nl_source_sets.values())
print("\nNum. sources:", len(nl_sources))
# Note that the number of distinct sources != the sum of contributions,
# because this count of contributions only concerns sources which are
# unique to a single dataset.

NL sources:

Unique sources contributed per dataset:
PolitiFact Fact Check: 631
Liar-PLUS: 488
AVeriTeC: 1766
FakeHealth: 431
Fake News Corpus: 187
ReCoVery: 1087
MultiFC: 238
MM-COVID19: 400
DRUID+: 174

Num. sources: 9422


#### Investigate and compare labels to decide on subsets of datasets to keep

Let's take a look at all the labels we have:

- [0, 1]
- Truth-o-meter
- Heterogenous
- {Refuted, Conflicting Evidence/Cherrypicking, Supported, Not Enough Evidence}
- {low, mixed, high}
- Factuality in {very high, high, mostly factual, mixed, low, very low}
- Credibility in {high, medium, low}
- [0, 5]
- {questionable, fake}
- Binary
- {false, true, unverified}

All labels except the following can at least implicitly be mapped to a scalar: 

- {false, true unverified}
- {Refuted, Conflicting Evidence/Cherrypicking, Supported, Not Enough Evidence},
- Heterogenous 

All of those except the last can at least partially implicitly be mapped to a scalar.  

Further, let us consider which of the labels come from dominant datasets.

- [0, 1] from Domain Ratings makes up almost all domains
- Truth-o-meter comes from PolitiFact Fact Check
- MultiFC contributes the large heterogeneous set of labels
- {Refuted, Conflicting Evidence/Cherrypicking, Supported, Not Enough Evidence} comes from AVeriTeC

This provokes the question of whether we can maybe pick out a useful subset of labels from MultiFC.

In [52]:
nl_sources_sans_multifc = [
    source_set
    for (name, source_set) in all_nl_source_sets.items()
    if name != "MultiFC"
]
unique_multifc_nl_sources = multifc_nl_sources.difference(
    set().union(*nl_sources_sans_multifc)
)
multifc_sources_df = multifc.to_pandas()
unique_multifc_sources_df = multifc_sources_df[
    multifc_sources_df["speaker"].str.lower().isin(unique_multifc_nl_sources)
]

unique_multifc_sources_df.groupby("label")["speaker"].nunique().sort_values(
    ascending=False
)

label
false                         43
mostly false                  33
mostly true                   33
half-true                     22
true                          21
factscan score: misleading    17
factscan score: true          16
factscan score: false         14
none                          13
a lot of baloney              11
a little baloney              11
2 pinnochios                   9
pants on fire!                 8
some baloney                   8
fact                           7
half true                      6
4 pinnochios                   6
misleading                     5
not the whole story            5
half flip                      5
fiction                        4
3 pinnochios                   3
full flop                      2
needs context                  1
unsupported                    1
Name: speaker, dtype: int64

In [53]:
# Given these observations, it's well-motivated to just extract the regular
# PolitiFact entries.

# We can retrieve the Truth-o-meter labels from this dataset
politifact_truth_o_meter = set(
    [
        s.replace("-", " ")
        for s in politifact_fact_check["verdict"]
        if s != "half-true"
    ]
)

speakers_from_multifc_from_politifact = set(
    str_list_to_lowercase(
        unique_multifc_sources_df[
            unique_multifc_sources_df["label"].isin(politifact_truth_o_meter)
        ]["speaker"].values
    )
)

speakers_from_multifc_from_politifact

{'adam bandt',
 'america’s last line of defense',
 'andrew leigh',
 'andrew robb',
 'anthony albanese',
 "atlanta mayor kasim reed's office",
 'australian labor party',
 'bob baldwin',
 'bob carr',
 'bob katter',
 'bornalivetruth.org',
 'bruce billson',
 'cathy mcgowan',
 'chris bowen',
 'chris everyman',
 'christine milne',
 'christopher pyne',
 'clive palmer',
 'conservativecolumns.com',
 'cory bernardi',
 'david coburn mep',
 'david leyonhjelm',
 'douglas holtz-eakin',
 'douglas ross mp',
 'election tv ads',
 'facebook image',
 'fiona patten',
 'fiona scott',
 'first minister nicola sturgeon',
 'gavin williamson, defence secretary',
 'george brandis',
 'getup!',
 'greg hunt',
 'heather wheeler mp',
 'humza yousaf',
 'ian blackford, snp westminster leader',
 'jamie briggs',
 'jason clare',
 'jenny macklin',
 'jeremy corbyn, labour leader',
 'jim torrance/all under one banner',
 'joe hockey',
 'john madigan',
 'john nicolson',
 'john redwood mp',
 'julia gillard',
 'julie bishop',
 'j

Another remaining question is whether we can map the AVeriTeC labels: {Refuted, Conflicting Evidence/Cherrypicking, Supported, Not Enough Evidence}. At first pass, Refuted may equal 0, Conflicting Evidence/Cherrypicking may equal something like 0.5, Supported may equal 1, and Not Enough Evidence may be mapped to something like 0.3 or dropped from the dataset.

In [54]:
# Supported + Refuted make up 2355/2792 speaker/label pairs.
averitec_df.groupby("label").count()

,claim,required_reannotation,justification,claim_date,speaker,original_claim_url,fact_checking_article,reporting_source,location_ISO_code,claim_types,fact_checking_strategies,questions,cached_original_claim_url,answer_types
label,,,,,,,,,,,,,,
Conflicting Evidence/Cherrypicking,233,233,233,222,180,105,233,180,201,233,233,233,77,233
Not Enough Evidence,317,317,317,313,257,161,317,246,286,317,317,317,144,317
Refuted,2047,2047,2047,1981,1534,1026,2047,1737,1818,2047,2047,2047,878,2047
Supported,971,971,971,947,821,483,971,717,877,971,971,971,357,971


Hence for now, we might just expect to map Refuted/Supported and drop remaining rows.

In [55]:
averitec_df = averitec_df[averitec_df["label"].isin(["Refuted", "Supported"])]
averitec_df

,claim,required_reannotation,label,justification,claim_date,speaker,original_claim_url,fact_checking_article,reporting_source,location_ISO_code,claim_types,fact_checking_strategies,questions,cached_original_claim_url,answer_types
0,"In a letter to Steve Jobs, Sean Connery refuse...",False,Refuted,The answer and sources show that the claim was...,31-10-2020,None,None,https://web.archive.org/web/20201130144023/htt...,Facebook,None,[Event/Property Claim],[Written Evidence],[{'answers': [{'answer': 'It was first publish...,None,"[Abstractive, Extractive]"
1,Trump Administration claimed songwriter Billie...,False,Refuted,Seems that the Wzshington post accused the sin...,31-10-2020,None,None,https://web.archive.org/web/20201103001419/htt...,Instagram,US,"[Position Statement, Event/Property Claim]",[Written Evidence],[{'answers': [{'answer': 'A Washington Post st...,None,[Extractive]
2,Due to Imran Khan's criticism of Macron's comm...,False,Refuted,The tweet was not the official government page...,31-10-2020,Consulate General Of Pakistan France,https://web.archive.org/web/20201113115127/htt...,https://web.archive.org/web/20210629013122/htt...,Twitter,FR,"[Causal Claim, Event/Property Claim]",[Written Evidence],[{'answers': [{'answer': 'Macron described Isl...,https://web.archive.org/web/20201113115127/htt...,"[Extractive, Extractive, Unanswerable, Unanswe..."
3,UNESCO declared Nadar community as the most an...,False,Refuted,This claim is refuted. According to the QA pai...,31-10-2020,Kumar Shankar,None,https://web.archive.org/web/20210225110220/htt...,Facebook,IN,[Event/Property Claim],[Written Evidence],"[{'answers': [{'answer': ""Nadar (also referred...",None,"[Extractive, Unanswerable, Extractive]"
4,Republican Matt Gaetz was part of a company th...,True,Refuted,The company was sold in 2004 and the law suit ...,31-10-2020,,None,https://web.archive.org/web/20210713185816/htt...,Facebook,US,"[Numerical Claim, Event/Property Claim]","[Written Evidence, Numerical Comparison]",[{'answers': [{'answer': 'No answer could be f...,None,"[Unanswerable, Extractive, Extractive, Abstrac..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3560,"In the UK in 2020 there were 48,000 illegal im...",True,Refuted,"48,000 is the total number of asylum seekers, ...",None,None,None,https://web.archive.org/web/20201101144602/htt...,Facebook,None,[Numerical Claim],"[Written Evidence, Consultation]",[{'answers': [{'answer': 'A National Audit Off...,None,"[Extractive, Extractive, Extractive]"
3561,5G wrecks immune systems and that is why peopl...,True,Refuted,Q&A 2 points out that 5g radio waves are non-i...,None,None,None,https://web.archive.org/web/20210616041324/htt...,Facebook,None,[Causal Claim],"[Consultation, Fact-checker Reference]",[{'answers': [{'answer': 'there is no evidence...,None,"[Abstractive, Abstractive, Abstractive, Abstra..."
3562,Bollywood star Shah Rukh Khan has slammed the ...,True,Refuted,Shah Rukh Khan did not slam the Modi governmen...,None,None,None,https://web.archive.org/web/20200806131752/htt...,None,IN,[Quote Verification],[Written Evidence],"[{'answers': [{'answer': 'orientaltimes.in', '...",None,"[Extractive, Abstractive, Unanswerable, Abstra..."
3563,Drinking one glass of water before bed prevent...,True,Supported,According to source provided it is medically p...,None,Nikkis Weight Loss Support,None,https://web.archive.org/web/20201227050241/htt...,Facebook,None,[Causal Claim],"[Written Evidence, Consultation]","[{'answers': [{'answer': 'No', 'answer_type': ...",None,"[Boolean, Extractive, Boolean, Boolean]"


### Datasets

Create dataframe that keeps label and provenance information.

In [56]:
nl_source_df = pd.DataFrame(
    columns=["id", "source_name", "label", "source_dataset"]
)


def merge_df_with_source_df(
    df,
    target_df,
    source_name_col: str,
    label_col: Union[str, List[str]],
    dataset_name: str,
    bias_col: str = None,
):
    idx_start = len(target_df)
    idx_end = idx_start + len(df)
    if isinstance(label_col, List):
        df["label"] = df.apply(lambda row: {col: row[col] for col in label_col}, axis=1)
        if bias_col:
            rename_mapping = {
                source_name_col: "source_name",
                bias_col: "bias",
                "label": "label",
            }
    elif bias_col:
        rename_mapping = {
            source_name_col: "source_name",
            label_col: "label",
            bias_col: "bias",
        }
    else:
        rename_mapping = {source_name_col: "source_name", label_col: "label"}
    df = df.rename(columns=rename_mapping)
    df = df[rename_mapping.values()]
    df["source_dataset"] = dataset_name
    df["id"] = np.arange(idx_start, idx_end)
    if not bias_col:
        df["bias"] = None
    return pd.concat([target_df, df])


nl_source_df = merge_df_with_source_df(
    politifact_fact_check.to_pandas(),
    nl_source_df,
    "statement_originator",
    "verdict",
    "PolitiFact Fact Check",
)

nl_source_df = merge_df_with_source_df(
    liar_plus.to_pandas(), nl_source_df, "Speaker", "Label", "Liar-PLUS"
)

# AveriTeC is the one dataset with two useful source columns. Let's split it. The merged DataFrame will be cleaned later.
averitec_speakers = averitec.to_pandas()[["speaker", "label"]]
nl_source_df = merge_df_with_source_df(
    averitec_speakers, nl_source_df, "speaker", "label", "AVeriTeC"
)
averitec_reporting_sources = averitec.to_pandas()[
    ["reporting_source", "label"]
]
nl_source_df = merge_df_with_source_df(
    averitec_reporting_sources,
    nl_source_df,
    "reporting_source",
    "label",
    "AVeriTeC",
)

nl_source_df = merge_df_with_source_df(
    fakehealth.to_pandas(), nl_source_df, "news_source", "rating", "FakeHealth"
)

nl_source_df = merge_df_with_source_df(
    fake_news_corpus.to_pandas(),
    nl_source_df,
    "authors",
    "type",
    "FakeNewsCorpus",
)

nl_source_df = merge_df_with_source_df(
    recovery.to_pandas(),
    nl_source_df,
    "publisher",
    "reliability",
    "ReCoVery",
    "political_bias",
)

recovery_authors_dataset = extract_dataset_with_authors_from_list_column(
    recovery, "author"
)
nl_source_df = merge_df_with_source_df(
    recovery_authors_dataset.to_pandas(),
    nl_source_df,
    "author",
    "reliability",
    "ReCoVery",
    "political_bias",
)

# Extract the subset of MultiFC we decided to keep.
multifc_politifact_factscan_subset = unique_multifc_sources_df[
    unique_multifc_sources_df["label"].isin(politifact_truth_o_meter)
]

nl_source_df = merge_df_with_source_df(
    multifc_politifact_factscan_subset,
    nl_source_df,
    "speaker",
    "label",
    "MultiFC",
)

nl_source_df = merge_df_with_source_df(
    mm_covid19.to_pandas(), nl_source_df, "agency", "label", "MM-COVID19"
)

mm_covid19_authors_dataset = extract_dataset_with_authors_from_list_column(
    mm_covid19, "authors"
)
nl_source_df = merge_df_with_source_df(
    mm_covid19_authors_dataset.to_pandas(),
    nl_source_df,
    "authors",
    "label",
    "MM-COVID19",
)

nl_source_df = merge_df_with_source_df(
    druid_plus_subset_df,
    nl_source_df,
    "claimant",
    "factcheck_verdict",
    "DRUID+"
)

nl_source_df["label"] = nl_source_df["label"].astype(str)

# Normalize source designation before saving the dataset.
nl_source_df["source_name"] = (
    nl_source_df["source_name"].str.lower().str.strip()
)

nl_sources_dataset = datasets.Dataset.from_pandas(nl_source_df).remove_columns(
    ["__index_level_0__"]
)
nl_sources_dataset

Dataset({
    features: ['id', 'source_name', 'label', 'source_dataset', 'bias'],
    num_rows: 65973
})

In [57]:
nl_sources_dataset.to_pandas()

,id,source_name,label,source_dataset,bias
0,0,barack obama,true,PolitiFact Fact Check,None
1,1,matt gaetz,false,PolitiFact Fact Check,None
2,2,kelly ayotte,mostly-true,PolitiFact Fact Check,None
3,3,bloggers,false,PolitiFact Fact Check,None
4,4,bobby jindal,half-true,PolitiFact Fact Check,None
...,...,...,...,...,...
65968,65968,"sherri tenpenny, paul joseph watson",False,DRUID+,None
65969,65969,"sherri tenpenny, paul joseph watson",False,DRUID+,None
65970,65970,"sherri tenpenny, paul joseph watson",False,DRUID+,None
65971,65971,"sherri tenpenny, paul joseph watson",False,DRUID+,None


In [58]:
nl_sources_dataset.to_pandas()["source_dataset"].unique()

array(['PolitiFact Fact Check', 'Liar-PLUS', 'AVeriTeC', 'FakeHealth',
       'FakeNewsCorpus', 'ReCoVery', 'MultiFC', 'MM-COVID19', 'DRUID+'],
      dtype=object)

Time to aggregate the datasets. Downstream, we want to query WikiData for each source only once. Hence we will do the following: Each source shall have a label column with a dict which maps each source dataset key with a list of ratings from that dataset to the source.

In [59]:
def aggregate_df_entries(
    df,
    source_col_name="source_name",
    source_dataset_col_name="source_dataset",
    label_col_name="label",
):
    """
    There is some very hairy Pandas logic here. The point is that each label
    given to the same source in the same dataset is aggregated in a list (1),
    and each such list is then collected in a dict whose key is the name of the
    source dataset (2).
    """
    # (1)
    df_grouped_by_name_and_dataset = (
        df.groupby([source_col_name, source_dataset_col_name])[label_col_name]
        .apply(list)
        .reset_index()
    )
    # (2)
    df_aggregated = (
        df_grouped_by_name_and_dataset.groupby(source_col_name)
        .apply(
            lambda row: dict(zip(row[source_dataset_col_name], row[label_col_name])),
            include_groups=False,
        )
        .reset_index(name="dataset_label_map")
    )
    # Recover the bias column
    return df_aggregated.merge(df.groupby(source_col_name).first(), on=source_col_name)


nl_source_df_aggregated = aggregate_df_entries(nl_source_df)
# Recover the bias column
nl_source_df_aggregated["id"] = np.arange(len(nl_source_df_aggregated))

## Domains

### Sets

In [60]:
domain_ratings_domains = set(domain_ratings["domain"])
news_media_reliability_emnlp_domains = set(
    news_media_reliability_emnlp["source_url_normalized"]
)
fake_news_corpus_domains = set(fake_news_corpus["urls"])

Domains extracted from URLs

In [ ]:
fakenewsnet_domains = extract_set_of_domains_from_urls(fakenewsnet["news_url"])
emergent_domains = extract_set_of_domains_from_urls(
    emergent["original_url_phase1"]
)
gossipcop_domains = extract_set_of_domains_from_urls(gossipcop["news_url"])
misinfo_general_domains = extract_set_of_domains_from_urls(
    misinfo_general_sources["representative_article_url"]
)
mm_covid19_domains = extract_set_of_domains_from_urls(
    mm_covid19["canonical_link"]
)

NameError: name 'misinfo_general_sources' is not defined

In [ ]:
all_domain_sets = {
    # Pure domains
    "Domain Ratings": domain_ratings_domains,
    "News Media Reliability EMNLP": news_media_reliability_emnlp_domains,
    "Fake News Corpus": fake_news_corpus_domains,
    # Extracted domains
    "FakeNewsNet": fakenewsnet_domains,
    "Emergent.Info": emergent_domains,
    "GossipCop": gossipcop_domains,
    "MisInfo General": misinfo_general_domains,
    "MM-COVID19": mm_covid19_domains,
}

domains = set().union(*all_domain_sets.values())

print("Domains:")
_print_num_unique_source_contributions_per_item(all_domain_sets)

print("\nNumber of unique domains:", len(domains))

Domains:

Unique sources contributed per dataset:
Domain Ratings: 9714
News Media Reliability EMNLP: 92
Fake News Corpus: 25
FakeNewsNet: 202
Emergent.Info: 41
GossipCop: 1526
MisInfo General: 37
MM-COVID19: 75

Number of unique domains: 13544


### Datasets

Create domain dataset that keeps label and dataset information.

In [63]:
news_media_reliability_emnlp.to_pandas()


,source_url,source_url_normalized,ref,fact,bias
0,http://www.villagevoice.com/,villagevoice.com,http://mediabiasfactcheck.com/the-village-voice/,high,left
1,https://insideclimatenews.org/,insideclimatenews.org,https://mediabiasfactcheck.com/insideclimate-n...,high,left-center
2,http://www.fury.news/,fury.news,http://mediabiasfactcheck.com/fury-news/,low,extreme-right
3,http://now8news.com/,now8news.com,http://mediabiasfactcheck.com/now8news/,low,center
4,http://constitution.com/,constitution.com,http://mediabiasfactcheck.com/the-constitution/,low,extreme-right
...,...,...,...,...,...
1061,http://fusion.kinja.com/,fusion.kinja.com,http://mediabiasfactcheck.com/fusion/,mixed,left-center
1062,http://aptnnews.ca,aptnnews.ca,https://mediabiasfactcheck.com/aptn-news/,high,center
1063,http://www.wsws.org,wsws.org,http://mediabiasfactcheck.com/world-socialist-...,high,left
1064,https://www.spin.com/death-and-taxes/,spin.com-death-and-taxes,http://mediabiasfactcheck.com/deathandtaxes/,high,left


In [ ]:
domain_df = pd.DataFrame(
    columns=["id", "source_name", "label", "source_dataset"]
)

# NOTE: Currently running with the PC1 ratings. This dataset contains multiple ratings per domain.
domain_df = merge_df_with_source_df(
    domain_ratings.to_pandas(), domain_df, "domain", "pc1", "Domain Ratings"
)

# Stick with the unnormalized URLs. That helps with WikiData crossreferencing later.
domain_df = merge_df_with_source_df(
    news_media_reliability_emnlp.to_pandas(),
    domain_df,
    "source_url_normalized",
    "fact",
    "News Media Reliability",
    "bias",
)

domain_df = merge_df_with_source_df(
    fake_news_corpus.map(
        lambda row: extract_domain_from_url_in_row(row, "urls")
    ).to_pandas(),
    domain_df,
    "urls",
    "type",
    "FakeNewsCorpus",
)

domain_df = merge_df_with_source_df(
    fakenewsnet.map(
        lambda row: extract_domain_from_url_in_row(row, "news_url")
    ).to_pandas(),
    domain_df,
    "news_url",
    "label",
    "FakeNewsNet",
)

domain_df = merge_df_with_source_df(
    emergent.map(
        lambda row: extract_domain_from_url_in_row(row, "original_url_phase1")
    ).to_pandas(),
    domain_df,
    "original_url_phase1",
    "fact_tag_phase1",
    "Emergent.Info",
)

domain_df = merge_df_with_source_df(
    gossipcop.map(
        lambda row: extract_domain_from_url_in_row(row, "news_url")
    ).to_pandas(),
    domain_df,
    "news_url",
    "label",
    "GossipCop",
)

domain_df = merge_df_with_source_df(
    misinfo_general_sources.map(
        lambda row: extract_domain_from_url_in_row(
            row, "representative_article_url"
        )
    ).to_pandas(),
    domain_df,
    "representative_article_url",
    ["factuality", "conspiracy", "pseudosci"],
    "MisInfo General",
    "bias"
)

domain_df = merge_df_with_source_df(
    mm_covid19.map(
        lambda row: extract_domain_from_url_in_row(row, "canonical_link")
    ).to_pandas(),
    domain_df,
    "canonical_link",
    "label",
    "MM-COVID19",
)

C:\Users\tharg\AppData\Local\Temp\ipykernel_14200\3336183133.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([target_df, df])


AssertionError: 

In [67]:
domain_df

,id,source_name,label,source_dataset,bias
0,0,reuters.com,1.0,Domain Ratings,None
1,1,apnews.com,0.998049,Domain Ratings,None
2,2,charitynavigator.org,0.985752,Domain Ratings,None
3,3,rollcall.com,0.982851,Domain Ratings,None
4,4,smithsonianmag.com,0.971184,Domain Ratings,None
...,...,...,...,...,...
1061,12581,fusion.kinja.com,mixed,News Media Reliability,left-center
1062,12582,aptnnews.ca,high,News Media Reliability,center
1063,12583,wsws.org,high,News Media Reliability,left
1064,12584,spin.com-death-and-taxes,high,News Media Reliability,left


At this point, let's quickly move domains that are easily spotted in the NL source set.

In [ ]:
# Ad hoc URL filtering. Not thorough.
@np.vectorize
def _is_probably_url(source_name: str) -> bool:
    return bool(
        source_name is not None
        and (domain_info := tldextract.extract(source_name)) is not None
        and (domain_info.subdomain or domain_info.suffix)
        and " " not in source_name
        and "." in source_name
    )


urls_from_nl_sources = nl_source_df_aggregated[
    _is_probably_url(nl_source_df_aggregated["source_name"].values)
].drop(columns=["dataset_label_map"])

domain_df = pd.concat([domain_df, urls_from_nl_sources])


# Return the subset of nl_sources_dataset that isn't in
# urls_from_nl_sources. Seems faster than built-in mapping-type
# filtering function. Uncertain if there is a more elegant way to do
# this.
mask = np.full_like(nl_source_df_aggregated["id"], fill_value=True, dtype=bool)
mask[urls_from_nl_sources["id"]] = False
nl_source_df_aggregated = nl_source_df_aggregated[mask]
# NOTE: When converting the dataset label map from Pandas to HuggingFace
# datasets, it enforces a consistent schema by adding datasetname-None
# pairs for each dataset that a source does not appear in. This makes it
# more computationally consistent but less readable, hence some local
# view should be used when investigating the set of maps.

nl_source_df_aggregated["dataset_label_map"] = nl_source_df_aggregated["dataset_label_map"].astype("string")
nl_sources_dataset = datasets.Dataset.from_pandas(nl_source_df_aggregated)
nl_sources_dataset.save_to_disk(os.path.join(DATA_DIR, "nl_sources_dataset"))
nl_sources_dataset.to_pandas()

Saving the dataset (0/1 shards):   0%|          | 0/9126 [00:00<?, ? examples/s]

,source_name,dataset_label_map,id,label,source_dataset,bias,__index_level_0__
0,,"{'AVeriTeC': ['Refuted', 'Refuted', 'Refuted',...",0,Refuted,AVeriTeC,None,0
1,"""crystal,",{'AVeriTeC': ['Refuted']},1,Refuted,AVeriTeC,None,1
2,"""watch the water""",{'PolitiFact Fact Check': ['pants-fire']},2,pants-fire,PolitiFact Fact Check,None,2
3,# 1 nwo hatr,"{'FakeNewsCorpus': ['questionable', 'questiona...",3,questionable,FakeNewsCorpus,None,3
4,",",{'ReCoVery': ['0']},4,0,ReCoVery,Extreme left,4
...,...,...,...,...,...,...,...
9121,අපි හෙළයෝ,{'AVeriTeC': ['Refuted']},9274,Refuted,AVeriTeC,None,9274
9122,රොබින් හුඩ් - robin hood,{'AVeriTeC': ['Refuted']},9275,Refuted,AVeriTeC,None,9275
9123,සතුට යනු කතොලිකයකු වීමයි,{'AVeriTeC': ['Refuted']},9276,Refuted,AVeriTeC,None,9276
9124,හොරොව්පතාන පාරම්පරික හෙළ වෙද ගෙදර,{'AVeriTeC': ['Refuted']},9277,Refuted,AVeriTeC,None,9277


Finish aggregating the domain dataset.

In [ ]:
# Be aware that the domains from Domain Ratings and News Media
# Reliability are incompletely formatted vis-a-vis WikiData URL
# entities, so an experimental approach must be adopted when
# cross-referencing those.
domain_df_aggregated = aggregate_df_entries(domain_df)

# A few newlines have snuck in which cause problems when forming queries later
domain_df_aggregated["dataset_label_map"] = (
    domain_df_aggregated["dataset_label_map"].astype(str).replace("\n", "")
)
domain_df_aggregated.reset_index(inplace=True)
domain_df_aggregated["label"] = domain_df_aggregated["label"].astype(str)

domain_df_aggregated
# domain_dataset = datasets.Dataset.from_pandas(domain_df_aggregated)
# domain_dataset.save_to_disk(os.path.join(DATA_DIR, "domain_dataset"))
# domain_dataset

,index,source_name,dataset_label_map,id,label,source_dataset,bias
0,0,0585news.com,{'Domain Ratings': [0.261046492905716]},10070,0.261046492905716,Domain Ratings,None
1,1,100fedup.com,{'Domain Ratings': [0.346782192375088]},9064,0.346782192375088,Domain Ratings,None
2,2,100giornidaleoni.it,{'Domain Ratings': [0.259866242365655]},10084,0.259866242365655,Domain Ratings,None
3,3,100milefreepress.net,{'Domain Ratings': [0.599200073403383]},5363,0.599200073403383,Domain Ratings,None
4,4,100percentfedup.com,"{'Domain Ratings': [0.21938033253327], 'News M...",10890,0.21938033253327,Domain Ratings,extreme-right
...,...,...,...,...,...,...,...
11607,11607,zonagrigianews.com,{'Domain Ratings': [0.328633442713656]},9223,0.328633442713656,Domain Ratings,None
11608,11608,zonazealots.com,{'Domain Ratings': [0.648645766965601]},3955,0.648645766965601,Domain Ratings,None
11609,11609,zootfeed.com,{'Domain Ratings': [0.216405500798777]},11004,0.216405500798777,Domain Ratings,None
11610,11610,zuerst.de,{'Domain Ratings': [0.675066817056849]},3329,0.675066817056849,Domain Ratings,None
